## Step 0 — Environment, configuration, reproducibility

All knobs live in one place so later steps cannot silently disagree with Step 1.

In [37]:
# ==============================================================
# Step 0 — environment, configuration, reproducibility
# ==============================================================
import os, sys, time, json, gc, random, platform, subprocess, warnings
from pathlib import Path
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

# ---- reproducibility -------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# ---- task constants (one source of truth for every later step) -------------
K           = 5                              # MAP@K, K = 5 as per the submission template
WATCH_FRAC  = 0.80                           # share of a programme's airtime that counts as "watched"
WATCH_MIN_S = 30                             # absolute guard: ignore overlaps shorter than this
VAL_START   = pd.Timestamp("2020-05-18 00:00:00")   # Monday of week 21  -> end of window A
TEST_START  = pd.Timestamp("2020-07-27 00:00:00")   # Monday of week 31  -> start of test window
VAL_START_S  = int(VAL_START.timestamp())           # the same instants as integer epoch seconds,
TEST_START_S = int(TEST_START.timestamp())          # the unit the loaders and the join work in

SCHED_FILES = ["export_arh_11-20-final.csv",   # weeks 11-20  -> window A
               "export_arh_21-30-final.csv",   # weeks 21-30  -> window B
               "export_arh_31-42-final.csv"]   # weeks 31-42  -> test schedule (airing times only)

# ---- data location ---------------------------------------------------------
def find_data_dir() -> Path:
    """Locate the competition CSVs on Kaggle, Colab, or a local machine."""
    cands = []
    if os.environ.get("SWEETTV_DATA"):
        cands.append(Path(os.environ["SWEETTV_DATA"]))
    cands += [Path("/kaggle/input/competitions/sweettv-tv-program-recommender"),
              Path("/kaggle/input/competitions/sweettv-tv-program-recommender"),
              Path("/kaggle/input"),
              Path("/content/sweettv-tv-program-recommender"),
              Path("/content"), Path("data"), Path("."),
              Path.home()/"Downloads"/"sweettv-tv-program-recommender"]
    for base in ("/kaggle/input", "/content"):
        if Path(base).is_dir():
            cands += [p for p in Path(base).iterdir() if p.is_dir()]
    for c in cands:
        if (c/"dataset11-30.csv").is_file():
            return c
    raise FileNotFoundError(
        "dataset11-30.csv not found. Put the four CSVs in ./data or set SWEETTV_DATA=/path/to/folder.")

DATA = find_data_dir()
WORK = Path(os.environ.get("SWEETTV_WORK", "sweettv_work"))
WORK.mkdir(parents=True, exist_ok=True)

def save_df(df: pd.DataFrame, name: str) -> Path:
    """Parquet when pyarrow is available, gzipped CSV otherwise."""
    try:
        p = WORK/f"{name}.parquet"; df.to_parquet(p, index=False); return p
    except Exception:
        p = WORK/f"{name}.csv.gz"; df.to_csv(p, index=False, compression="gzip"); return p

gpu = "none"
try:
    import torch
    if torch.cuda.is_available():
        gpu = f"{torch.cuda.get_device_name(0)} (CUDA {torch.version.cuda})"
except Exception:
    pass

print(f"python      : {platform.python_version()} on {platform.system()} {platform.release()}")
print(f"pandas      : {pd.__version__}   numpy: {np.__version__}")
try:
    import psutil
    ram = f"{psutil.virtual_memory().total/1e9:.1f} GB"
except Exception:
    ram = "unknown (psutil not installed)"
print(f"CPU cores   : {os.cpu_count()}   RAM: {ram}")
print(f"GPU         : {gpu}")
print(f"data dir    : {DATA}")
print(f"work dir    : {WORK.resolve()}")
print(f"seed={SEED}  K={K}  WATCH_FRAC={WATCH_FRAC}  WATCH_MIN_S={WATCH_MIN_S}s")

python      : 3.13.15 on Linux 6.18.55+
pandas      : 2.3.3   numpy: 2.1.3
CPU cores   : 4   RAM: 33.7 GB
GPU         : Tesla T4 (CUDA 12.8)
data dir    : /kaggle/input/competitions/sweettv-tv-program-recommender
work dir    : /kaggle/working/sweettv_work
seed=42  K=5  WATCH_FRAC=0.8  WATCH_MIN_S=30s


## Step 1 — Data model and the join rule

### 1.1 What the two tables are

`dataset11-30.csv` is a **view log**: one row per continuous viewing interval of one user on one channel.
After removing 17 exact duplicates it has 5,821,970 rows, ids are unsigned 64-bit, and `duraton` (sic) is in seconds. No programme id is present.

The three `export_arh_*` files are a **programme schedule**: `(channel_id, start_time, duration)` plus the programme that aired in that slot. Only ~11 % of slots carry a `tv_show_id`; the rest are unlabelled airtime (`tv_show_id = 0`).

A "user watched programme $i$" statement therefore has to be *constructed*: it is true when the user's viewing interval on channel $c$ **overlaps** the airing interval of $i$ on the same channel $c$, by enough of the programme's length.

### 1.2 The join rule

For a view $v = (user, c, [v_s, v_e))$ and an airing $p = (c, [p_s, p_e))$ on the same channel:

$$\text{overlap}(v,p) = \max(0,\ \min(v_e, p_e) - \max(v_s, p_s))$$

$$v \text{ counts as watching } p \iff \text{overlap}(v,p) \ge W \quad\text{and}\quad \frac{\text{overlap}(v,p)}{p_e - p_s} \ge F$$

* **Channel equality is part of the key**, not a heuristic filter: a programme airing on channel 3 says nothing about what a user watched on channel 5 at the same minute.
* Time overlap, not containment: users join mid-programme and leave early, and the log's median view (396 s) is much shorter than the labelled-slot median duration (2,700 s), so containment would throw away almost everything.
* **Two different lengths are in play and must never be swapped.** The log's `duraton` is the *viewing interval* (median 396 s); the denominator of the fraction is the *programme's* `duration` from the schedule (median 2,700 s). Dividing by the view length instead would make the fraction identically 1 and turn every channel-zap into a "watch".
* The interval end is `start_time + duraton`, not `stop_time`. On the 2020-03-29 DST day (71,094 views start that day) the wall-clock difference between `stop_time` and `start_time` can be an hour longer than the elapsed `duraton`. The code uses `duraton`, the elapsed viewing length, so no hour of viewing is invented.
* `duration` from the schedule is the authoritative *airing* length (`p_e = p_s + duration`).

### 1.3 Threshold: fraction of the programme, not absolute seconds

I use **F = 0.80 of the programme's duration** with a 30 s absolute guard (W). The justification is empirical and is printed by the profiling cell below:

* An absolute-seconds cut does not transfer across slot lengths: the labelled schedule spans 60 s to 86,400 s slots (10th percentile 600 s, median 2,700 s). A 10-minute cut means "42 % of a sitcom" but "11 % of a film" and "100 % of a news bulletin".
* The task's own target is defined by the organiser as *"screen time (watched time / total duration of the show) ≥ 0.8"*. Using the same rule for the interaction table makes my validation target the same object the leaderboard measures, which is the fairest possible choice.
* 0.80 is strict but affordable: it still yields ~570 k interactions over ~4.8 k users (median ≈ 74 qualifying airings per user), so no user is starved by the strictness. The guard `W = 30 s` is below the shortest labelled slot (60 s) and is held at 30 s in every row of the sweep table; it only protects against very short overlaps if the schedule changes.
* The sensitivity table in §1.9 shows how the graph grows at F = 0.5 … 0.9, so the choice is auditable rather than asserted.

### 1.4 `tv_show_id = 0`

Dropped. It is **unlabelled airtime, not an item**: it has no id, so unlabelled rows cannot be identified with one another. Those slots are removed from the item inventory *before* the join, so a view that only touches unlabelled airtime produces no interaction, while a view that overlaps both an unlabelled slot and a labelled programme still matches the labelled programme. They are not treated as negative feedback ("misses"), because absence of a label is not evidence of dislike.

### 1.5 Interaction unit and leakage guard

* One row per **qualifying airing**, because "frequency of high-screen-time airings" is exactly the label-generating signal. Aggregation to `(user, show)` happens afterwards and keeps `n_watch` (the frequency), `watched_s`, `frac_mean`, and the first/last timestamp.
* The view log ends **2020-07-26 23:58:42**, i.e. before `TEST_START`. The test-window schedule (`export_arh_31-42`) is loaded, but the code physically splits it off (`sched_test`) and the join only ever sees `sched_train = sched[start < TEST_START]`. There is no code path that joins test-period airings to views.
* Windows are assigned by the **view's own start time**, not by which file a schedule row came from (the files overlap by a few hours around Monday 18 May 2020, so file-based assignment would leak a morning of week 21 into window A).

### 1.6 Loading

The loaders are deliberately thin: dtypes are pinned (the `user_id` column is a real `uint64`, so it must not be parsed as float — that silently corrupts ids above 2^53), timestamps are parsed explicitly, string columns are stored as `category` to keep the 3.4 M-row schedule inside a few hundred MB, and windows are materialised as integer seconds for the join.

In [38]:
# ==============================================================
# 1.6  Loaders
# ==============================================================
ISO_FMT = "%Y-%m-%d %H:%M:%S"
SLOT_FMT = "%d.%m.%Y %H:%M:%S"

def load_views(path=DATA/"dataset11-30.csv") -> pd.DataFrame:
    """View log -> integer-second intervals. One row per view."""
    # `stop_time` is deliberately NOT in usecols: the column is never read, so no
    # later cell can accidentally use the DST-skewed clock arithmetic.
    v = pd.read_csv(path, encoding="utf-8-sig",
                    usecols=["user_id", "vsetv_id", "start_time", "duraton"],
                    dtype={"user_id": "uint64", "vsetv_id": "int64", "duraton": "int64"})
    n_raw = len(v)
    v = v.drop_duplicates()                      # 17 exact duplicate rows in the raw file
    v["view_start"] = pd.to_datetime(v["start_time"], format=ISO_FMT).values.astype("datetime64[s]").astype("int64")
    # The interval end is ALWAYS start + duraton. `stop_time` is never read.
    # On 2020-03-29 (DST spring-forward) the logged clocks shift by one hour, so
    # stop_time - start_time overstates 117 views by exactly 3,600 s. Those
    # phantom hours would create false overlaps with the following programme.
    # `duraton` is the elapsed viewing length and is immune to the clock change.
    v["view_end"] = v["view_start"] + v["duraton"]
    v.attrs["n_raw"] = n_raw
    v.attrs["n_dst_day"] = int(v["start_time"].str.startswith("2020-03-29").sum())
    assert (v["duraton"] > 0).all(), "a view with non-positive duration cannot form an interval"
    assert (v["view_end"] > v["view_start"]).all()
    return v.rename(columns={"vsetv_id": "channel_id"})[
        ["user_id", "channel_id", "view_start", "view_end", "duraton"]]

SCHED_DTYPES = {"channel_id": "int64", "duration": "int64", "tv_show_id": "int64",
                "year_of_production": "string"}
SCHED_COLS = ["channel_id", "channel_title", "start_time", "duration", "tv_show_title",
              "tv_show_id", "tv_show_category", "tv_show_genre_1", "tv_show_genre_2",
              "tv_show_genre_3", "year_of_production", "director", "actors"]

# each file owns the calendar window it describes; used only to break (rare) duplicate keys
OWNERSHIP = {"export_arh_11-20-final.csv": ("2020-03-08", "2020-05-18"),
             "export_arh_21-30-final.csv": ("2020-05-18", "2020-07-27"),
             "export_arh_31-42-final.csv": ("2020-07-27", "2020-10-20")}

def load_schedule(files=SCHED_FILES) -> pd.DataFrame:
    """Schedule -> integer-second airing intervals, one row per airing."""
    frames = []
    for f in files:
        s = pd.read_csv(DATA/f, encoding="utf-8-sig", low_memory=False,
                        usecols=SCHED_COLS, dtype=SCHED_DTYPES)
        s["source_file"] = f
        frames.append(s)
    s = pd.concat(frames, ignore_index=True)

    s["slot_start"] = pd.to_datetime(s["start_time"], format=SLOT_FMT)
    s["sched_start"] = s["slot_start"].values.astype("datetime64[s]").astype("int64")
    s["sched_end"]   = s["sched_start"] + s["duration"]

    # ownership priority -> deterministic de-duplication of identical airings
    prio = pd.Series(np.full(len(s), len(OWNERSHIP), dtype="int8"))
    for i, (f, (lo, hi)) in enumerate(OWNERSHIP.items()):
        inside = ((s["slot_start"] >= pd.Timestamp(lo)) & (s["slot_start"] < pd.Timestamp(hi))
                  & (s["source_file"] == f))
        prio = prio.mask(inside, i)
    s = (s.assign(_prio=prio)
          .sort_values(["_prio", "channel_id", "sched_start"])
          .drop_duplicates(["channel_id", "sched_start", "tv_show_id", "duration"], keep="first")
          .drop(columns="_prio")
          .reset_index(drop=True))

    for c in ["channel_title", "tv_show_title", "tv_show_category", "tv_show_genre_1",
              "tv_show_genre_2", "tv_show_genre_3", "director", "actors", "source_file"]:
        s[c] = s[c].astype("category")
    return s

t0 = time.time()
views = load_views()
sched = load_schedule()
print(f"loaded in {time.time()-t0:.1f}s")
print(f"views   {len(views):,} rows after dropping {views.attrs['n_raw']-len(views)} exact duplicates"
      f" | {views.user_id.nunique():,} users | {views.channel_id.nunique():,} channels")
print(f"        interval end = start + duraton (stop_time never read); "
      f"{views.attrs['n_dst_day']:,} views start on the 2020-03-29 DST day")
print(f"schedule {len(sched):,} airings | {sched.channel_id.nunique():,} channels | "
      f"{sched.tv_show_id.nunique():,} distinct show ids (incl. 0)")

# ---- the leakage guard: the join only ever sees airings before the test window ----
sched_train = sched[sched.sched_start <  TEST_START_S].copy()
sched_test  = sched[sched.sched_start >= TEST_START_S].copy()
views["window"] = np.where(views.view_start < VAL_START_S, "A_11_20", "B_21_30")

assert views.view_start.max() < TEST_START_S, "view log must not contain test-period behaviour"
assert sched_train.sched_start.max() < TEST_START_S
assert sched_test.sched_start.min() >= TEST_START_S
print(f"train airings {len(sched_train):,} | test airings {len(sched_test):,} (test = airing times only)")

loaded in 37.5s
views   5,821,970 rows after dropping 17 exact duplicates | 4,914 users | 202 channels
        interval end = start + duraton (stop_time never read); 71,094 views start on the 2020-03-29 DST day
schedule 3,444,007 airings | 405 channels | 9,672 distinct show ids (incl. 0)
train airings 2,146,200 | test airings 1,297,807 (test = airing times only)


### 1.7 Profiling the raw material

Every number the threshold argument depends on is produced here, from the files themselves.

In [39]:
# ==============================================================
# 1.7  Profile: durations, coverage, label density
# ==============================================================
def q(x, ps=(.1, .25, .5, .75, .9, .99)):
    return {f"p{int(p*100)}": round(float(np.quantile(x, p)), 1) for p in ps}

v_dur = views.duraton.to_numpy()
lab = sched_train[sched_train.tv_show_id != 0]

print("VIEW LOG")
print(f"  date range            : {pd.to_datetime(views.view_start.min(), unit='s')}  ->  "
      f"{pd.to_datetime(views.view_start.max(), unit='s')}")
print(f"  view duration (s)     : min {v_dur.min():.0f}  {q(v_dur)}  max {v_dur.max():,.0f}")
print(f"  views per user        : median {views.groupby('user_id').size().median():.0f}, "
      f"p90 {views.groupby('user_id').size().quantile(.9):.0f}")

print("\nSCHEDULE (training weeks 11-30)")
print(f"  airings               : {len(sched_train):,}   channels: {sched_train.channel_id.nunique()}")
print(f"  labelled airings      : {len(lab):,}  ({len(lab)/len(sched_train):.1%} of slots)")
print(f"  labelled slot dur (s) : min {lab.duration.min():.0f}  {q(lab.duration.to_numpy())}  max {lab.duration.max():,.0f}")
print(f"  distinct shows        : {lab.tv_show_id.nunique():,}")
titles_per_show = lab.groupby("tv_show_id", observed=True).tv_show_title.nunique()
print(f"  shows with >1 title   : {(titles_per_show>1).sum():,} / {len(titles_per_show):,}"
      "  -> item metadata must be aggregated over airings, not read from one row")

print("\nCHANNEL OVERLAP BETWEEN THE TWO TABLES")
vc, sc = set(views.channel_id.unique()), set(sched_train.channel_id.unique())
print(f"  view channels {len(vc)}, schedule channels {len(sc)}, shared {len(vc & sc)}")
print(f"  views on channels absent from schedule: {views.channel_id.isin(vc - sc).sum():,} "
      f"({views.channel_id.isin(vc - sc).mean():.2%}) -> lost by construction")

print("\nWHY A FRACTION CUT RATHER THAN AN ABSOLUTE ONE")
print(f"  median view {np.median(v_dur):.0f}s vs median labelled slot {lab.duration.median():.0f}s"
      f"  -> a fixed 10-min cut is 42% of a 1,440s slot but 150% of a 400s slot")

VIEW LOG
  date range            : 2020-03-09 00:00:02  ->  2020-07-26 23:58:42
  view duration (s)     : min 61  {'p10': 80.0, 'p25': 133.0, 'p50': 396.0, 'p75': 1452.0, 'p90': 3911.0, 'p99': 14079.3}  max 342,145
  views per user        : median 874, p90 2430

SCHEDULE (training weeks 11-30)
  airings               : 2,146,200   channels: 404
  labelled airings      : 236,851  (11.0% of slots)
  labelled slot dur (s) : min 60  {'p10': 600.0, 'p25': 1200.0, 'p50': 2700.0, 'p75': 5400.0, 'p90': 7200.0, 'p99': 13800.0}  max 32,700
  distinct shows        : 8,233
  shows with >1 title   : 3,495 / 8,233  -> item metadata must be aggregated over airings, not read from one row

CHANNEL OVERLAP BETWEEN THE TWO TABLES
  view channels 202, schedule channels 404, shared 197
  views on channels absent from schedule: 113,578 (1.95%) -> lost by construction

WHY A FRACTION CUT RATHER THAN AN ABSOLUTE ONE
  median view 396s vs median labelled slot 2700s  -> a fixed 10-min cut is 42% of a 1,440s slo

### 1.8 The join, implemented as an exact sweep

A naive `views × schedule` cross-join is 5.8 M × 2.1 M rows (training-window airings). Instead each channel's labelled airings are sorted by start time and queried with `np.searchsorted`, walking backwards only while a prefix-maximum of end times still reaches into the view. This is the standard **interval stabbing** pattern: it returns *exactly* the same pair set as brute force (asserted below on a random sample), in O(n log m + candidates) time.

The sample check matters: it is the difference between "the loop probably found the right rows" and "the optimised join is provably equal to the definition".

In [40]:
# ==============================================================
# 1.8  Interval-overlap join (exact, per channel)
# ==============================================================
def overlap_join(views: pd.DataFrame, sched: pd.DataFrame,
                 min_frac: float = WATCH_FRAC, min_sec: float = WATCH_MIN_S,
                 channels: set | None = None) -> pd.DataFrame:
    """Join views to labelled airings on (channel, positive time overlap).

    Returns one row per qualifying airing:
      user_id, tv_show_id, channel_id, overlap_s, frac, show_dur,
      air_start, view_start, view_end
    """
    lab = sched[(sched.tv_show_id != 0) & (sched.duration > 0)]
    out = []
    for ch, sv in lab.groupby("channel_id", observed=True):
        if channels is not None and ch not in channels:
            continue
        sv = sv.sort_values("sched_start")
        st  = sv.sched_start.to_numpy(); en = sv.sched_end.to_numpy()
        sid = sv.tv_show_id.to_numpy();  dur = sv.duration.to_numpy()
        emax = np.maximum.accumulate(en)                 # for the backward scan bound

        vv = views[views.channel_id == ch]
        if vv.empty:
            continue
        vs  = vv.view_start.to_numpy(); ve = vv.view_end.to_numpy()
        uid = vv.user_id.to_numpy()
        hi  = np.searchsorted(st, ve, side="right")      # airings that start before the view ends

        for k in range(len(vv)):
            i = hi[k] - 1
            while i >= 0 and emax[i] > vs[k]:            # stop once no earlier airing can reach the view
                if en[i] >= vs[k]:
                    ov = min(ve[k], en[i]) - max(vs[k], st[i])
                    if ov > 0:
                        frac = ov / dur[i]
                        if frac >= min_frac and ov >= min_sec:
                            out.append((uid[k], sid[i], ch, ov, frac, dur[i], st[i], vs[k], ve[k]))
                i -= 1
    cols = ["user_id", "tv_show_id", "channel_id", "overlap_s", "frac",
            "show_dur", "air_start", "view_start", "view_end"]
    return pd.DataFrame(out, columns=cols)


def brute_force_channel(views_ch, sched_ch, min_frac=WATCH_FRAC, min_sec=WATCH_MIN_S):
    """Reference implementation: vectorised all-pairs overlap for ONE channel."""
    lab = sched_ch[(sched_ch.tv_show_id != 0) & (sched_ch.duration > 0)]
    rows = []
    for _, v in views_ch.iterrows():
        ov = (np.minimum(v.view_end, lab.sched_end.to_numpy())
              - np.maximum(v.view_start, lab.sched_start.to_numpy()))
        keep = (ov > 0) & (ov / lab.duration.to_numpy() >= min_frac) & (ov >= min_sec)
        for j in np.flatnonzero(keep):
            rows.append((int(v.user_id), int(lab.tv_show_id.iloc[j]), int(lab.channel_id.iloc[j]),
                         int(ov[j]), float(ov[j]/lab.duration.iloc[j])))
    return pd.DataFrame(rows, columns=["user_id","tv_show_id","channel_id","overlap_s","frac"])

# ---- provable equality on a random sample of channels ------------------------
rng = np.random.default_rng(SEED)
sampled_ch = rng.choice(sorted(set(views.channel_id) & set(sched_train.channel_id)), 12, replace=False)
fast = overlap_join(views, sched_train, channels=set(sampled_ch))
slow = pd.concat([brute_force_channel(views[views.channel_id == c], sched_train[sched_train.channel_id == c])
                  for c in sampled_ch], ignore_index=True)
key = ["user_id","tv_show_id","channel_id","overlap_s","frac"]
a = fast[key].copy(); b = slow[key].copy()
for df in (a, b):
    df["user_id"] = df["user_id"].astype("uint64")   # ids exceed int64, so never go through int64
ga = a.groupby(key, as_index=False).size().sort_values(key).reset_index(drop=True)
gb = b.groupby(key, as_index=False).size().sort_values(key).reset_index(drop=True)
assert ga.equals(gb), "optimised join does not reproduce brute force"
print(f"join equality check passed: {len(fast):,} interactions on {len(sampled_ch)} sampled channels "
      f"({', '.join(map(str, sampled_ch[:6]))}, ...) - identical to brute force")

join equality check passed: 18,708 interactions on 12 sampled channels (644, 1517, 1168, 803, 1030, 1294, ...) - identical to brute force


### 1.9 Running the join: sensitivity, final table, coverage

The sweep is not hyper-parameter tuning — it documents *what the threshold does to the graph*, so the single choice used everywhere can be defended. `F = 0.80` is then fixed for the rest of the notebook.

In [41]:
# ==============================================================
# 1.9  Sensitivity, final join, aggregation, coverage
# ==============================================================
# ---- how much does the fraction rule actually filter? ----------------------
def candidate_stats(views, sched):
    """All positive view-airing overlaps, with no threshold: the raw material of the join."""
    lab = sched[(sched.tv_show_id != 0) & (sched.duration > 0)]
    n_pairs = 0; n_views = 0; n_hit = 0; fracs = []
    for ch, sv in lab.groupby("channel_id", observed=True):
        vv = views[views.channel_id == ch]
        if vv.empty:
            continue
        sv = sv.sort_values("sched_start")
        st, en, dur = sv.sched_start.to_numpy(), sv.sched_end.to_numpy(), sv.duration.to_numpy()
        emax = np.maximum.accumulate(en)
        vs, ve = vv.view_start.to_numpy(), vv.view_end.to_numpy()
        hi = np.searchsorted(st, ve, side="right")
        n_views += len(vv)
        for k in range(len(vv)):
            i = hi[k] - 1; hit = 0
            while i >= 0 and emax[i] > vs[k]:
                if en[i] >= vs[k]:
                    ov = min(ve[k], en[i]) - max(vs[k], st[i])
                    if ov > 0:
                        fracs.append(ov / dur[i]); n_pairs += 1; hit += 1
                i -= 1
            n_hit += hit > 0
    return n_views, n_hit, n_pairs, np.array(fracs)

t0 = time.time()
n_views, n_hit, n_pairs, fracs = candidate_stats(views, sched_train)
print(f"views on schedule channels            : {n_views:,}")
print(f"  ... with >=1 labelled overlap       : {n_hit:,} ({n_hit/n_views:.1%})"
      f"  -> {n_views-n_hit:,} views hit only unlabelled airtime and create no interaction")
print(f"candidate (view, airing) pairs        : {n_pairs:,}  ({n_pairs/n_hit:.2f} per matching view)")
print(f"  overlap fraction p50 {np.quantile(fracs,.5):.2f} | p75 {np.quantile(fracs,.75):.2f} | "
      f"p90 {np.quantile(fracs,.9):.2f} | p99 {np.quantile(fracs,.99):.2f}")
print("  candidate pairs surviving each fraction cut: " +
      ", ".join(f"F={f:.1f}: {(fracs>=f).mean():.1%}" for f in [0.5, 0.6, 0.7, 0.8, 0.9]))
print(f"candidate scan {time.time()-t0:.1f}s")

# ---- threshold sensitivity: what the cut does to the graph -----------------
t0 = time.time()
print("\nthreshold sensitivity (channel+overlap join, labelled airings only)")
print(f"{'F':>5} {'min_s':>6} {'interactions':>13} {'users':>7} {'shows':>7} "
      f"{'med/view-user':>14} {'users>=5shows':>14}")
sweep = []
for f in [0.50, 0.60, 0.70, 0.80, 0.90]:
    j = overlap_join(views, sched_train, min_frac=f, min_sec=WATCH_MIN_S)
    per_u = j.groupby("user_id").size()
    cov5 = j.groupby("user_id").tv_show_id.nunique().ge(5).mean()
    sweep.append(dict(F=f, min_s=WATCH_MIN_S, interactions=len(j), users=j.user_id.nunique(),
                      shows=j.tv_show_id.nunique(), med_per_user=float(per_u.median()),
                      users_ge5=cov5))
    print(f"{f:>5.2f} {WATCH_MIN_S:>6} {len(j):>13,} {j.user_id.nunique():>7,} {j.tv_show_id.nunique():>7,} "
          f"{per_u.median():>14.0f} {cov5:>13.1%}")
print(f"sweep runtime {time.time()-t0:.1f}s")

# ---- THE join used by every later step --------------------------------------
t0 = time.time()
pairs = overlap_join(views, sched_train, min_frac=WATCH_FRAC, min_sec=WATCH_MIN_S)
pairs["air_dt"] = pd.to_datetime(pairs.air_start, unit="s")
print(f"\nfinal join: {len(pairs):,} (view, airing) pairs in {time.time()-t0:.1f}s")

# one interaction = one *airing* watched, not one viewing session:
# a user who zapped away during a film and came back still watched that airing once
airings = (pairs.sort_values("overlap_s", ascending=False)
                .drop_duplicates(["user_id", "tv_show_id", "air_start"])
                .copy())
airings["window"] = np.where(airings.view_start < VAL_START_S, "A_11_20", "B_21_30")
print(f"collapsed to {len(airings):,} distinct (user, show, airing) events "
      f"({len(pairs)-len(airings):,} repeat sessions on an already-counted airing)")

# ---- aggregate to (user, show) ---------------------------------------------
user_show = (airings.groupby(["user_id", "tv_show_id"], observed=True)
             .agg(n_watch=("air_start", "size"),                 # frequency = the label's own signal
                  watched_s=("overlap_s", "sum"),                # seconds actually viewed
                  frac_mean=("frac", "mean"),
                  show_dur=("show_dur", "median"),
                  n_channels=("channel_id", "nunique"),
                  first_ts=("air_start", "min"),
                  last_ts=("air_start", "max"))
             .reset_index())
print(f"user x show matrix: {len(user_show):,} pairs "
      f"({len(user_show)/(user_show.user_id.nunique()*user_show.tv_show_id.nunique()):.3%} dense)")

# ---- data-quality note: very long "channel left on" sessions ----------------
# A view is a viewing interval, so a 40-hour view legitimately contains ~50 programmes.
# Because the denominator is the *programme* duration, each of those gets frac = 1.0.
airings["view_len"] = airings.view_end - airings.view_start
print("\ninteractions contributed by long views (the view, not the programme, is long):")
for h in (3, 6, 12, 24):
    m = airings.view_len > h*3600
    print(f"  views >{h:>2}h : {m.sum():>7,} interactions ({m.mean():6.2%}) "
          f"from {airings.loc[m,'user_id'].nunique():>5,} users | "
          f"{airings.loc[m,'view_len'].sum()/airings.view_len.sum():.1%} of matched viewing time")
share_long = (airings.assign(m=airings.view_len > 6*3600)
                     .groupby("user_id").m.mean())
print(f"  longest view {airings.view_len.max()/3600:.1f}h; "
      f"{int((share_long>0.5).sum())} users get >50% of their interactions from views >6h")
print("  -> the organiser's rule counts these, so we keep them; Step 2 tests the sensitivity to it")

# ---- coverage / sanity ------------------------------------------------------
sub = pd.read_csv(DATA/"submission.csv", dtype={"user_id": "uint64"})
sub_users = set(sub.user_id)
u_cnt = user_show.groupby("user_id").agg(n_shows=("tv_show_id", "nunique"),
                                         n_watch=("n_watch", "sum"))
print(f"\nsubmission users                        : {len(sub_users):,}")
print(f"  with >=1 interaction (weeks 11-30)    : {len(sub_users & set(u_cnt.index)):,}")
print(f"  with >=5 distinct programmes          : {len(sub_users & set(u_cnt.index[u_cnt.n_shows>=5])):,}")
print(f"  no usable history (cold start)        : {len(sub_users - set(u_cnt.index)):,}")

for w in ["A_11_20", "B_21_30"]:
    aw = airings[airings.window == w]
    uw = user_show[user_show.user_id.isin(aw.user_id.unique())]
    print(f"window {w}: {len(aw):>7,} airings | {aw.user_id.nunique():>5,} users | "
          f"{aw.tv_show_id.nunique():>5,} shows | median shows/user "
          f"{uw.groupby('user_id').tv_show_id.nunique().median():.0f}")

# does the item set transfer from the validation target to the real test window?
test_shows = set(sched_test.loc[sched_test.tv_show_id != 0, "tv_show_id"])
aw_b = airings[airings.window == "B_21_30"]
print(f"B airings on programmes that also air in weeks 31-42 : {aw_b.tv_show_id.isin(test_shows).mean():.1%}")
print(f"B programmes present in the A inventory               : {aw_b.tv_show_id.isin(set(airings[airings.window=='A_11_20'].tv_show_id)).mean():.1%}")

# ---- leakage assertions -----------------------------------------------------
assert airings.air_start.max() < TEST_START_S, "an interaction was built from a test-window airing"
assert airings.view_start.max() < TEST_START_S
assert airings.tv_show_id.ne(0).all()
assert (airings.frac >= WATCH_FRAC).all() and (airings.overlap_s >= WATCH_MIN_S).all()
assert set(airings.channel_id) <= set(sched_train.channel_id)

# ---- persist for the next steps --------------------------------------------
p1 = save_df(pairs.drop(columns=["air_dt"]), "interactions_raw")   # (view, airing) provenance
p2 = save_df(airings.drop(columns=["air_dt"]), "airings")          # one row per watched airing
p3 = save_df(user_show, "user_show")
items = (sched[sched.tv_show_id != 0]
         .groupby("tv_show_id", observed=True)
         .agg(n_airings=("sched_start", "size"),
              n_train_airings=("sched_start", lambda x: int((x < TEST_START_S).sum())),
              n_test_airings=("sched_start", lambda x: int((x >= TEST_START_S).sum())),
              med_dur=("duration", "median"),
              title=("tv_show_title", lambda x: x.mode().iloc[0] if len(x.mode()) else ""),
              category=("tv_show_category", lambda x: x.mode().iloc[0] if len(x.mode()) else ""))
         .reset_index())
p4 = save_df(items, "items")
print(f"\nsaved: {p1}\n       {p2}\n       {p3}\n       {p4}")
print(f"items: {len(items):,} distinct programmes, "
      f"{int((items.n_test_airings>0).sum()):,} of them air in weeks 31-42")

views on schedule channels            : 4,718,037
  ... with >=1 labelled overlap       : 2,964,334 (62.8%)  -> 1,753,703 views hit only unlabelled airtime and create no interaction
candidate (view, airing) pairs        : 3,847,664  (1.30 per matching view)
  overlap fraction p50 0.12 | p75 0.45 | p90 1.00 | p99 1.00
  candidate pairs surviving each fraction cut: F=0.5: 23.0%, F=0.6: 19.8%, F=0.7: 17.2%, F=0.8: 14.8%, F=0.9: 12.8%
candidate scan 10.6s

threshold sensitivity (channel+overlap join, labelled airings only)
    F  min_s  interactions   users   shows  med/view-user  users>=5shows
 0.50     30       885,019   4,866   4,050            129         97.5%
 0.60     30       761,859   4,859   3,992            107         96.7%
 0.70     30       659,912   4,846   3,922             89         95.9%
 0.80     30       570,577   4,834   3,817             74         94.8%
 0.90     30       491,921   4,818   3,705             60         92.9%
sweep runtime 64.8s

final join: 570,577 (

## Step 1 — results and what they mean

**Read-out** *(from the Kaggle run whose outputs are saved in this notebook; the pipeline is deterministic)*

1. **The join is well covered.** 4,718,037 views fall on a channel that appears in the schedule; 2,964,334 of them (62.8 %) overlap at least one labelled airing. The remaining 1,753,703 views (37.2 %) land only on unlabelled airtime and create no interaction. This is the labelling caveat, not a join bug.
2. **Most airtime is not a "watch".** 3,847,664 candidate (view, airing) pairs exist (1.30 per matching view). Their median overlap fraction is 0.12, and 14.8 % survive F = 0.8.
3. **The final interaction table.** 570,577 (view, airing) pairs give 562,919 distinct watched airings (7,658 repeat sessions on an airing already counted), and 207,950 `(user, show)` pairs. That is 1.13 % dense over 4,834 users × 3,817 programmes (207,950 / 18,451,378). The median is 74 airings per user, and 94.8 % of users have at least 5 distinct programmes.
4. **Windows A and B are balanced**: 282,913 vs 280,006 airings and 4,726 vs 4,623 users. 94.1 % of B airings are on programmes that also air in weeks 31–42. 85.5 % of B programmes are present in the A inventory, so 14.5 % (100 − 85.5) are not, and only a content-aware method can rank them. Restricting the pool to the target window's schedule is therefore not an artificial handicap.
5. **Cold start is small.** 1,255 of 1,260 submission users have usable history, 1,238 have at least 5 programmes, and 5 have none. Those 5 receive the shared fallback.
6. **The test pool includes programmes that never air in weeks 11–30.** The inventory has 9,671 programmes, of which 6,646 air in weeks 31–42. A pool drawn from training history alone would make the others unrecommendable, so the pool is the labelled inventory of the target window's schedule.
7. **Data-quality handling is explicit.** (a) The interval is rebuilt as `start_time + duraton` (§1.2). (b) A view can last up to 95.0 h while the denominator is the programme, so 216,891 of 562,919 interactions (38.53 %) come from views longer than 3 h, and 74,597 (13.25 %) from views longer than 6 h. 88 users get more than 50 % of their interactions from views longer than 6 h ("channel left on"). The organiser's rule counts these, so the main table keeps them; §2.10 tests the sensitivity.

**Runtime and environment (this run).** Python 3.13.15, pandas 2.3.3, NumPy 2.1.3, 4 CPU cores, 33.7 GB RAM, Kaggle. A Tesla T4 is visible, but no code path uses it. Loading took 37.5 s, the candidate scan 10.6 s, the threshold sweep 64.8 s and the final join 12.6 s.

**Handover to Step 2**

| Artefact | Contents | Used by |
|---|---|---|
| `interactions_raw` | 570,577 rows, one per (view, airing), provenance for every interaction | audit / reproducibility |
| `airings` | 562,919 rows, one per watched airing, tagged `A_11_20` / `B_21_30` | PPR graph, recency features, validation target |
| `user_show` | 207,950 rows: `n_watch`, `watched_s`, `frac_mean`, first/last timestamp | content profiles, popularity baseline |
| `items` | 9,671 programmes with train/test airing counts, modal title and category | candidate pool, content features |

**Next (Step 3):** content-based scoring over the pooled schedule, using title, category, genres, decade, director and cast from the schedule only; then personalised PageRank on the item-item co-view graph as the second method. Both get the same split, pool, fallback and ranker, and both are compared against the popularity reference above.

## Step 2 — Problem definition: windows, pool, repeats, fallback, metric

This block fixes every definition the later methods must share. Nothing here fits a model; it defines *what a method is allowed to see*.

### 2.1 Join rule and "watched" threshold (implemented in Step 1, restated)

A view is matched to a programme when **same channel** *and* **positive time overlap** with the airing, with

$$\frac{\text{overlap}}{\text{programme duration}} \ge 0.80 \quad\text{and}\quad \text{overlap} \ge 30\,\text{s}.$$

The denominator is the **programme's** duration (schedule `duration`, median 2,700 s), never the view's `duraton` (median 396 s). The fraction is used rather than an absolute-seconds cut because labelled slots span 60 s–32,700 s in the training window: a fixed 10-minute rule is "42 % of a sitcom" but "11 % of a film", whereas 0.8 means the same thing for every slot. 0.8 is also the organiser's own definition, which makes the validation target the same object the leaderboard scores. The 30 s guard is a no-op on this data (0 interactions removed) and exists only to protect against degenerate slots.

### 2.2 `tv_show_id = 0` (implemented in Step 1, restated)

**Dropped as an item class, not used as context, not treated as a miss.** Those rows are unlabelled airtime: they have no id and do not identify a programme. They are removed from the inventory *before* matching, so a view that touches only unlabelled airtime yields nothing (1,753,703 views, 37.2 %), while a view that also touches a labelled airing still counts for it. Treating them as negative feedback would be wrong: an absent label is not evidence of dislike, it is absence of evidence.

### 2.3 Candidate pool

$$\mathcal{P}(\text{window}) = \{\, \text{tv\_show\_id} \ne 0 \;:\; \text{the programme airs at least once inside that window} \,\}$$

The pool is the **labelled inventory of the target window's schedule**, identical in construction for validation (weeks 21–30) and for the test (weeks 31–42). Three reasons:

* A programme that does not air in the target window cannot be watched in it, so it can never be a label. Any hit it produced would waste one of the five slots.
* The target is *defined* over the target window's schedule, so this pool is a guaranteed superset of the label set — by construction, not by luck.
* It is the only use of the test schedule the brief permits: "which programme airs on which channel at which time". No viewing data from the test window exists or is touched.

Pool sizes: **5,842** (weeks 11–20), **6,376** (weeks 21–30, used for validation), **6,646** (weeks 31–42, used for submission), against a training-schedule inventory of 8,233.

**Decision on the test-window programmes that never air in weeks 11–30: keep them in the pool.** Removing them would guarantee misses on every slot they could have filled. They are not featureless: the schedule carries title, category, genre, year and cast fields, so the content method can describe and rank them from metadata alone. Behaviourally they are cold — nobody in weeks 11–30 watched them — so the co-view graph has no edge to them and personalised PageRank cannot give them any mass. That is a **structural limitation of the graph method**, not a reason to shrink the pool; the head-to-head table reports how often each method reaches these behaviourally cold items. The only use of the test schedule remains "which programme airs on which channel at which time"; no test-window viewing data is read or exists.

**Repeats: allowed, in every method.** A programme the user already watched may be recommended again. The competition does not forbid it, and the data makes the choice one-sided: **53.1 % of the programmes in a user's weeks 21–30 top-5 were already in their weeks 11–20 history**, and 14.8 % of users repeat their entire top-5. Filtering history out would throw away more than half of the correct answers. The *only* shared constraint is that no method may remove a pool item on the grounds that the user watched it; methods compete on how they rank, not on which subset they are allowed to consider.

### 2.4 Validation split

| Window | Dates | Role |
|---|---|---|
| **A** weeks 11–20 | 2020-03-09 → 2020-05-18 | fit |
| **B** weeks 21–30 | 2020-05-18 → 2020-07-27 | score (target built with the organiser's rule) |
| Test weeks 31–42 | 2020-07-27 → 2020-10-19 | submission |

Justification:

* **Same protocol as the real task**: fit on ten weeks, predict the *next* window's top-5 by screen-time ≥ 0.8 → frequency → strictly 5. The test window is twelve weeks, so the forecast horizon is the same order of magnitude.
* **Temporal order preserved, no shuffling.** Every target event is strictly after every history event; the `Problem` object below asserts it.
* **The labels are the competition's labels.** B's target is built with the same join, the same 0.8 rule and the same 5-per-user cut as the private target, so MAP@5 on B measures the actual metric rather than a proxy.
* **It is not degenerate**: 282,913 vs 280,006 watched airings, 4,726 vs 4,623 users, and 14.5 % of B programmes (100 % − 85.5 %) are absent from the A inventory — the hold-out contains genuine cold items.
* **Evaluation population**: 3,923 users have ≥ 5 distinct B programmes (the organiser guarantees exactly 5 per test user); 3,868 of them also have A history. The headline MAP@5 is on the **3,868 warm users**, because that is the closest analogue of the all-warm test set. The 55 users with B activity but no A history are reported separately as a cold-start check.

**A caveat that belongs in the write-up:** for **83.6 %** of validation users the 5th frequency value is tied (median tie group 5 programmes, up to 64). The organiser's "strictly 5 most frequent" rule must have broken those ties arbitrarily, and so must we: target order is frequency DESC, then the user's last view start for that programme, then id. That puts a noise ceiling on MAP@5 that no model can remove, and it means small MAP differences between methods are not meaningful.

### 2.5 Training-only guarantee

Every method receives a `Problem` and may read only `history` (interactions strictly before `target_start`) and `pool`. It never sees `target`; the harness owns the target. `Problem` asserts `history.last_ts.max() < target_start` and `set(target) ⊆ pool`, so a method cannot be fitted on validation labels even by accident, and a prediction outside the pool is a hard error rather than a silent zero.

### 2.6 Output contract and fallback

Every method returns **exactly five** ids per user:

1. score every pool item, rank by `score DESC`, then recency of the user's last watch of it, then popularity, then id (deterministic);
2. take the first five not already taken;
3. if fewer than five remain — which only happens for a user with no usable history — pad with the **global popularity top-5 of the same pool**.

The fallback is a property of the harness, not of a method, so it is identical everywhere: **5 test users** hit it in the real submission, and **55 validation users** exercise it in the hold-out.

In [42]:
# ==============================================================
# 2.7  Shared problem definition, target builder, MAP@5
# ==============================================================
from dataclasses import dataclass

@dataclass
class Problem:
    """Everything a method may see. `target` is used by the harness only."""
    name: str
    history: pd.DataFrame            # user_id, tv_show_id, n_watch, last_ts  (all < target_start)
    pool: np.ndarray                 # candidate tv_show_id (sorted, int64)
    target_start: int
    target: dict | None = None       # user_id -> list of 5 ids (validation only)
    users: list | None = None        # scoring population when target is None (test)

    def __post_init__(self):
        assert self.history.last_ts.max() < self.target_start, "history contains target-window behaviour"
        assert len(np.unique(self.pool)) == len(self.pool)
        if self.target is not None:
            missing = set().union(*map(set, self.target.values())) - set(self.pool.tolist())
            assert not missing, f"{len(missing)} target items outside the pool"
            self.users = sorted(int(u) for u in self.target)
        else:
            assert self.users is not None, "a target-less problem must list its scoring users"
            self.users = sorted(int(u) for u in self.users)

        # Cached once: make_predictions() must not rebuild these sets per user.
        self._warm_users = {int(u) for u in self.history.user_id}
        self._pool_set = set(self.pool.tolist())

    @property
    def warm_users(self):
        return self._warm_users


def build_pool(sched: pd.DataFrame, start: int, end: int) -> np.ndarray:
    """Legal candidates: labelled, positive-length airings whose slot starts in [start, end)."""
    m = ((sched.tv_show_id != 0) & (sched.duration > 0)
         & (sched.sched_start >= start) & (sched.sched_start < end))
    return np.sort(sched.loc[m, "tv_show_id"].unique()).astype("int64")


def build_history(airings: pd.DataFrame, cutoff: int) -> pd.DataFrame:
    """Events are attributed to the view's start time (the same rule Step 1 uses for windows)."""
    h = airings[airings.view_start < cutoff]
    return (h.groupby(["user_id", "tv_show_id"], observed=True)
             .agg(n_watch=("air_start", "size"), last_ts=("view_start", "max"))
             .reset_index())


def build_target(airings: pd.DataFrame, start: int, end: int, k: int = K):
    """The organiser's rule: frequency of high-screen-time airings, strictly k, DESC."""
    w = airings[(airings.view_start >= start) & (airings.view_start < end)]
    g = (w.groupby(["user_id", "tv_show_id"], observed=True)
          .agg(n_watch=("air_start", "size"), last_ts=("view_start", "max"))
          .reset_index())
    g = g.sort_values(["user_id", "n_watch", "last_ts", "tv_show_id"],
                      ascending=[True, False, False, True])
    top = g.groupby("user_id").head(k)
    tgt = {u: [int(x) for x in ids] for u, ids in top.groupby("user_id").tv_show_id}
    return {u: l for u, l in tgt.items() if len(l) == k}, g


def map_at_k(pred: dict, target: dict, k: int = K):
    """MAP@k with the standard denominator min(|target|, k)."""
    aps = []
    for u, gt in target.items():
        p = pred.get(u, [])[:k]
        hits, s = 0, 0.0
        for i, x in enumerate(p):
            if x in gt:
                hits += 1
                s += hits / (i + 1)
        aps.append(s / min(len(gt), k))
    return float(np.mean(aps)), len(aps)


def rank_items(scores: pd.Series, n: int = K, tiebreak: dict | None = None) -> list:
    """score DESC, then tie-break DESC (recency/popularity), then id ASC - fully deterministic."""
    df = scores.rename("score").reset_index()
    df.columns = ["tv_show_id", "score"]
    df["tb"] = df.tv_show_id.map(tiebreak).fillna(0) if tiebreak else 0
    df = df.sort_values(["score", "tb", "tv_show_id"], ascending=[False, False, True])
    return df.tv_show_id.head(n).tolist()


def make_predictions(problem: Problem, scorer, fallback: list, k: int = K) -> dict:
    """Output contract: exactly k ids per user, identical for every method."""
    users = problem.users if problem.target is None else problem.target
    out = {}
    for u in users:
        ranked = scorer(u)[:k] if u in problem.warm_users else []
        if len(ranked) < k:                                   # cold start or short list
            ranked = ranked + [i for i in fallback if i not in ranked]
        out[u] = ranked[:k]
    return out

### 2.8 Building the two problems

`VAL` fits on A and scores on B. `TEST` fits on A ∪ B and has no target; its pool is the weeks 31–42 schedule.

In [43]:
# ==============================================================
# 2.8  Build pools, history, validation target; check the guarantees
# ==============================================================
P_A  = build_pool(sched, 0,            VAL_START_S)
P_B  = build_pool(sched, VAL_START_S,  TEST_START_S)
P_T  = build_pool(sched, TEST_START_S, 2**40)

hist_A = build_history(airings, VAL_START_S)      # weeks 11-20
hist_AB = build_history(airings, TEST_START_S)    # weeks 11-30
target_B, target_B_full = build_target(airings, VAL_START_S, TEST_START_S)

VAL  = Problem("validation A->B", hist_A,  P_B, VAL_START_S, target_B)
TEST = Problem("test  A+B->31-42", hist_AB, P_T, TEST_START_S, None,
               users=sorted(int(u) for u in sub_users))

print(f"pool sizes        : A {len(P_A):,} | B {len(P_B):,} | test {len(P_T):,}")
print(f"training inventory: {sched[sched.tv_show_id!=0].tv_show_id.nunique():,} programmes")
print(f"history A         : {len(hist_A):,} (user, show) pairs, {hist_A.user_id.nunique():,} users")
print(f"history A+B       : {len(hist_AB):,} pairs, {hist_AB.user_id.nunique():,} users")
print(f"target B          : {len(target_B):,} users x 5 ids "
      f"(from {target_B_full.user_id.nunique():,} users with any B activity)")

warm = VAL.warm_users & set(target_B)
cold = set(target_B) - VAL.warm_users
print(f"evaluable users   : {len(warm):,} warm + {len(cold):,} cold = {len(target_B):,}")

# --- target-structure diagnostics -------------------------------------------
ties = []
for u, d in target_B_full.groupby("user_id", sort=False):
    if len(d) < K:
        continue
    v = np.sort(d.n_watch.to_numpy())[::-1]
    ties.append((u, int((d.n_watch == v[K-1]).sum()), len(d)))
ties = pd.DataFrame(ties, columns=["user_id", "tied_at_k", "n_distinct"])
print(f"\ntie at the 5th frequency value: {(ties.tied_at_k>1).sum():,} / {len(ties):,} users "
      f"({(ties.tied_at_k>1).mean():.1%}), median tie group {ties.loc[ties.tied_at_k>1,'tied_at_k'].median():.0f}")

# --- how much does repetition matter? ---------------------------------------
A_items = hist_A.groupby("user_id").tv_show_id.apply(set).to_dict()
rep_item = np.mean([np.mean([i in A_items[u] for i in ids]) for u, ids in target_B.items() if u in A_items])
rep_user = np.mean([set(ids) <= A_items[u] for u, ids in target_B.items() if u in A_items])
print(f"B top-5 items already in the user's A history : {rep_item:.1%}")
print(f"users whose whole B top-5 was in A history    : {rep_user:.1%}")

# --- guarantees -------------------------------------------------------------
assert VAL.history.last_ts.max() < VAL_START_S
assert TEST.history.last_ts.max() < TEST_START_S
assert set().union(*map(set, target_B.values())) <= set(P_B.tolist())
print("\nleakage guards: history strictly before target window (VAL, TEST) | all targets inside pool B")

pool sizes        : A 5,842 | B 6,376 | test 6,646
training inventory: 9,671 programmes
history A         : 100,586 (user, show) pairs, 4,726 users
history A+B       : 207,950 pairs, 4,834 users
target B          : 3,923 users x 5 ids (from 4,623 users with any B activity)
evaluable users   : 3,868 warm + 55 cold = 3,923

tie at the 5th frequency value: 3,281 / 3,923 users (83.6%), median tie group 5
B top-5 items already in the user's A history : 53.1%
users whose whole B top-5 was in A history    : 14.8%

leakage guards: history strictly before target window (VAL, TEST) | all targets inside pool B


### 2.9 Reference baselines (not the two required methods)

The brief requires a popularity reference that every method must beat. I add one more reference — the user's own top-5 from A — because the repeat statistic above says habit is a strong signal, and a method that cannot beat "same again" is not adding anything. Both are **user-blind or history-only**, so they set the floor, not the ambition.

In [44]:
# ==============================================================
# 2.9  Popularity baseline, cold-start fallback, reference MAP@5
# ==============================================================
def popularity_scores(history: pd.DataFrame, pool: np.ndarray) -> pd.Series:
    """Distinct users who watched each programme; restricted to the pool."""
    pop = history.groupby("tv_show_id").user_id.nunique()
    return pop[pop.index.isin(set(pool.tolist()))]


# --- reference 1: global popularity (user-independent) -----------------------
pop_A   = popularity_scores(hist_A, P_B)                 # fitted on A only, restricted to pool B
pop_tie = hist_A.groupby("tv_show_id").agg(last=("last_ts", "max"))
POP_RANK = rank_items(pop_A, K, tiebreak=pop_tie["last"].to_dict())
FALLBACK = list(POP_RANK)                                 # same list for every method and every user

# --- reference 2: the user's own top-5 from A --------------------------------
own_sorted = hist_A[hist_A.tv_show_id.isin(P_B)].sort_values(["user_id", "n_watch", "last_ts", "tv_show_id"],
                                ascending=[True, False, False, True])
OWN_TOP = (own_sorted.groupby("user_id", sort=False).head(K)
                     .groupby("user_id").tv_show_id.apply(lambda s: [int(x) for x in s]).to_dict())

pred_pop = make_predictions(VAL, lambda u: POP_RANK, FALLBACK)
pred_own = make_predictions(VAL, lambda u: OWN_TOP.get(u, []), FALLBACK)

rows = []
for name, pred in [("popularity (reference)", pred_pop), ("own A-history top-5 (reference)", pred_own)]:
    m_warm, _ = map_at_k({u: pred[u] for u in warm}, {u: target_B[u] for u in warm})
    m_all, _  = map_at_k(pred, target_B)
    rows.append(dict(model=name, map5_warm=round(m_warm, 4), map5_all=round(m_all, 4)))
print(pd.DataFrame(rows).to_string(index=False))
print(f"\nfallback list (global popularity, pool B): {FALLBACK}")
print(f"users relying on it: {len(cold):,} cold validation users | 5 cold test users")

save_df(target_B_full, "val_target_full")
save_df(pd.DataFrame([(u, r, i) for u, ids in target_B.items() for r, i in enumerate(ids)],
                     columns=["user_id", "rank", "tv_show_id"]), "val_target")
save_df(hist_A, "val_history")
print("saved validation artefacts")

                          model  map5_warm  map5_all
         popularity (reference)     0.2368    0.2357
own A-history top-5 (reference)     0.3263    0.3240

fallback list (global popularity, pool B): [2400480, 240081, 2400467, 200344, 2400508]
users relying on it: 55 cold validation users | 5 cold test users
saved validation artefacts


### 2.10 Sensitivity: does the "channel left on" artefact change the conclusions?

Step 1 showed that 13.3 % of interactions come from views longer than 6 h. The headline table keeps them, because the organiser's rule does. This check rebuilds history **and target** from views ≤ 6 h and repeats the two references. It is a robustness test, not the headline: if the ranking of methods were an artefact of long views, it would show up here.

In [45]:
# ==============================================================
# 2.10  Robustness of the baseline to very long views (>6h dropped)
# ==============================================================
SHORT_MAX_S = 6 * 3600
air_short = airings[airings.view_len <= SHORT_MAX_S]

hist_A_s = build_history(air_short, VAL_START_S)
target_B_s, _ = build_target(air_short, VAL_START_S, TEST_START_S)
VAL_s = Problem("validation A->B, views<=6h", hist_A_s, P_B, VAL_START_S, target_B_s)
warm_s = VAL_s.warm_users & set(target_B_s)

pop_s = popularity_scores(hist_A_s, P_B)
POP_S = rank_items(pop_s, K, tiebreak=hist_A_s.groupby("tv_show_id").last_ts.max().to_dict())
own_sorted_s = hist_A_s[hist_A_s.tv_show_id.isin(P_B)].sort_values(["user_id", "n_watch", "last_ts", "tv_show_id"],
                                    ascending=[True, False, False, True])
OWN_S = (own_sorted_s.groupby("user_id", sort=False).head(K)
                       .groupby("user_id").tv_show_id.apply(lambda s: [int(x) for x in s]).to_dict())

pred_pop_s = make_predictions(VAL_s, lambda u: POP_S, POP_S)
pred_own_s = make_predictions(VAL_s, lambda u: OWN_S.get(u, []), POP_S)

rows = []
for name, main, short in [("popularity", pred_pop, pred_pop_s),
                          ("own A-history top-5", pred_own, pred_own_s)]:
    m_main, _  = map_at_k({u: main[u] for u in warm},   {u: target_B[u] for u in warm})
    m_short, _ = map_at_k({u: short[u] for u in warm_s}, {u: target_B_s[u] for u in warm_s})
    rows.append(dict(reference=name, map5_views_le_6h=round(m_short, 4), map5_all_views=round(m_main, 4),
                     delta=round(m_short - m_main, 4)))
print(f"interactions kept: {len(air_short)/len(airings):.2%}; "
      f"users: main {len(warm):,} | <=6h {len(warm_s):,}")
print(pd.DataFrame(rows).to_string(index=False))

common = [u for u in warm if u in target_B_s]
same   = np.mean([set(target_B[u]) == set(target_B_s[u]) for u in common])
print(f"\ntarget sets identical for {same:.1%} of the {len(common):,} comparable users "
      f"-> long views move the labels, so this variant is a different problem, not just cleaner data")
print("conclusion: the reference ordering is unchanged; the headline keeps all views to match the organiser's rule")

interactions kept: 86.75%; users: main 3,868 | <=6h 3,842
          reference  map5_views_le_6h  map5_all_views   delta
         popularity            0.2304          0.2368 -0.0064
own A-history top-5            0.3214          0.3263 -0.0049

target sets identical for 74.2% of the 3,843 comparable users -> long views move the labels, so this variant is a different problem, not just cleaner data
conclusion: the reference ordering is unchanged; the headline keeps all views to match the organiser's rule


## Step 2 — results and what they commit us to

| Decision | Value | Consequence for Steps 3–5 |
|---|---|---|
| Pool | target-window labelled inventory | 6,376 (val) / 6,646 (test) candidates per user |
| Repeats | allowed | no method may filter watched items |
| Fit window (val) | weeks 11–20 | methods see 4,726 users |
| Target window (val) | weeks 21–30 | 3,868 warm users scored, 55 cold |
| Metric | MAP@5, denominator `min(|target|, 5)` | order matters, ties at rank 5 cap the ceiling |
| Fallback | global popularity top-5 of the same pool | identical for every method |
| Reference | popularity **0.2368** MAP@5 (warm) | the number both real methods must beat |
| Second reference | own A-history top-5 **0.3263** MAP@5 | habit is strong; this is the bar that matters |

The popularity baseline is not a straw man: it already gets ~24 % of a perfect score because Ukrainian TV is habitual. The own-history reference does better — **0.3263** — because 53.1 % of the B top-5 is a repeat of something the user already watched in A. Any method that cannot beat both is not worth deploying, whatever its K.

**Long-view sensitivity (§2.10).** Dropping views > 6 h removes 13.25 % of interactions and changes the target set for 25.8 % of users. Popularity moves 0.2368 → 0.2304 and own-history loses ≈ 0.005: **the ordering is unchanged and both references lose about the same amount**, so the long-view artefact inflates the absolute level slightly but is not driving the comparison. The headline keeps all views to match the organiser's rule; every later method is compared on exactly that table.

**Runtime (Kaggle run: 4 CPU cores, Tesla T4 visible, no GPU code path).** Step 1 loading took 37.5 s, the threshold sweep 64.8 s and the final join 12.6 s. Each max-pooling content fit takes about 13 s and each PPR fit about 14 s (their `seconds` columns). Everything above is deterministic (sorted orders, no sampling), so a re-run reproduces the same numbers.

## Step 3 — Method 1: content-based filtering

### 3.0 The split both methods share

Nothing in Step 3 or Step 4 redefines the data. Both methods receive the same two `Problem` objects built in §2.8 and are fitted and scored by the same harness.

| | **validation** | **test / submission** |
|---|---|---|
| Fit window | A: weeks 11–20 (2020-03-09 → 2020-05-18) | A ∪ B: weeks 11–30 (→ 2020-07-27) |
| Target window | B: weeks 21–30 | weeks 31–42 |
| Candidate pool | labelled inventory of the **B** schedule — 6,376 ids | labelled inventory of the **test** schedule — 6,646 ids |
| Users scored | 3,868 warm + 55 cold = 3,923 | 1,260 submission users (5 cold) |
| Target visible to a method? | **no** | **no** (does not exist) |
| Fallback | global popularity top-5 of pool B | global popularity top-5 of pool T |
| Metric | MAP@5, denominator `min(|target|, 5)` | — |

In the wording of the brief: **A is the training window, B is the validation window, weeks 31–42 are the test window.** Both methods call `fit(problem.history, problem.target_start)` and rank `problem.pool`. Neither receives `problem.target`; `make_predictions` owns the target and applies the fallback. The only extra input is **schedule metadata** for content features, which is not behaviour: validation uses rows with `sched_start < TEST_START_S` (A + B schedules) and test uses the full schedule (`< 2**40`). The test schedule is the one future artefact the brief explicitly allows, and it is used only for "which programme airs when", never for "who watched what".

### 3.1 What content can and cannot do here

The hold-out has a structural property that makes a content model worth building: **14.5 % of B programmes are absent from the A inventory**, and **some test-pool programmes never air in weeks 11–30 at all**. A purely behavioural model has no edge to those items and can only rank them by a tie-break. A content model can describe them from the schedule itself — title, category, genre, decade, director, cast — and score them even with zero interactions.

The price is that content similarity is a proxy for taste, not evidence of it. Two programmes with the same genre label are not interchangeable, and the metadata is noisy (episode-level titles, missing genres). So this method is expected to be **broad but shallow**: it should reach cold items and score something for every user, but it may lose to history-based signals on the many users whose next week looks like their last month.

### 3.2 Item representation (schedule metadata only)

| Block | Fields | Vectorisation | Weight |
|---|---|---|---|
| Title | `tv_show_title` | word 1- and 2-grams, TF-IDF, `min_df=2`, sublinear tf, L2 row-normalised | 1.0 |
| Metadata | `tv_show_category`, `tv_show_genre_1..3`, decade of `year_of_production` | multi-hot over category + up to three genres + decade, L2 row-normalised | 0.7 |
| People | `director` + `actors` | token TF-IDF (no bigrams), `min_df=2`, L2 row-normalised | 0.4 |

Each block is L2-normalised **before** concatenation, then the concatenated vector is L2-normalised again, so a block's weight is its share of the cosine, not an artefact of its dimensionality. The weights are chosen on validation, not on test.

Metadata is aggregated per `tv_show_id` with the **modal value** over schedule rows before the feature cut-off. This matters: 3,495 of the 8,233 training programmes appear with more than one title (episode titles), so taking the first row would make the representation order-dependent. Modal aggregation is deterministic (ties broken by pandas' `value_counts` order, which is stable for a fixed frame). Genre fields are sparse, especially the second and third genre; a missing genre contributes nothing rather than a special "unknown" token, because "unknown" is not a genre.

No interaction count enters the item vector. Two programmes that air on the same channel are **not** similar by construction; only their metadata is.

### 3.3 User profile: weighted max-pooling, not a centroid

For user $u$ with fit-window history $H_u$, each watched programme $j$ gets

$$w_{uj} = \log(1+n_{uj})\; 2^{-\Delta_{uj}/h},$$

where $n_{uj}$ is the number of A airings watched and $\Delta_{uj}$ is the number of days between the user's last A watch of $j$ and the target start (2020-05-18). The logarithm stops a single heavily repeated programme from dominating; the half-life $h$ (28 or 84 days, or no decay, in the grid) lets a user's recent diet outweigh their spring habits.

The item score is **max-pooling** over the watched items:

$$s(u,i) = \max_{j \in H_u}\; w_{uj}\,\cos(x_i, x_j).$$

The ablation is the centroid profile $s(u,i)=\cos\!\big(x_i,\sum_j w_{uj}x_j\big)$. A user's history is multi-modal — children's TV in the morning, films at night, news at lunch — and a centroid averages those modes into a vector that resembles none of them. On the full validation population (§3.8) max-pooling reaches **0.2672** warm MAP@5, and the two centroid variants reach **0.1395** and **0.1406**. The centroid is kept in the grid as the ablation, not as the headline.

### 3.4 Ranking and determinism

For every user the pool is ordered by **score DESC → recency of that user's last watch DESC → global popularity DESC → programme id ASC**. The first three keys are the same for both methods; the last key makes ties reproducible. There is no sampling anywhere in this step, so two runs with the same inputs produce the same five ids. `SEED` only controls the environment, not the model.

### 3.5 Selection protocol

The grid varies the block weights, the half-life and the pooling rule. **Every fit uses A only and every score uses B.** The test window is not touched until §4.8, after the configuration is frozen. Selection is the maximum warm MAP@5; ties go to the earlier (simpler) row. The grid deliberately contains the centroid and a no-decay variant, so the table is also the evidence that the chosen components matter rather than an assertion.

### 3.6 Cost

Feature construction is one pass over the schedule plus six sparse TF-IDF/one-hot blocks. The expensive operation is the max-pooling scoring matrix (`pool × items`, float32), computed once per configuration. The per-user ranking is a `lexsort` over 6,376 floats. The whole grid is CPU sparse/dense linear algebra; the notebook runs unchanged on a GPU runtime, and the environment banner reports whether a GPU is present. No CUDA kernel is required for this method.

In [46]:
# ==============================================================
# 3.0  Split audit: the two problems both methods will share
# ==============================================================
FEATURE_END_VAL  = TEST_START_S     # validation may use the A+B schedule metadata
FEATURE_END_TEST = 2 ** 40          # test may use the full schedule metadata

SPLIT_AUDIT = pd.DataFrame([
    dict(problem=p.name,
         fit_events=len(p.history),
         fit_users=int(p.history.user_id.nunique()),
         last_fit_event=pd.to_datetime(int(p.history.last_ts.max()), unit="s"),
         target_start=pd.to_datetime(p.target_start, unit="s"),
         score_users=len(p.users),
         pool=len(p.pool),
         metadata_window=("A+B schedule" if p is VAL else "all schedule"))
    for p in (VAL, TEST)
])
print(SPLIT_AUDIT.to_string(index=False))
for p in (VAL, TEST):
    assert p.history.last_ts.max() < p.target_start
    assert len(p.users) > 0
print("\nboth methods receive exactly these objects: fit on `history`, rank `pool`; target is harness-only")

         problem  fit_events  fit_users      last_fit_event target_start  score_users  pool metadata_window
 validation A->B      100586       4726 2020-05-17 23:59:33   2020-05-18         3923  6376    A+B schedule
test  A+B->31-42      207950       4834 2020-07-26 23:19:53   2020-07-27         1260  6646    all schedule

both methods receive exactly these objects: fit on `history`, rank `pool`; target is harness-only


### 3.7 Item features and the content model (self-contained, no scikit-learn)

The code below is the whole method: tokenisation, TF-IDF, multi-hot blocks, modal metadata, the weighted profile and the deterministic ranker. It uses only `numpy`, `pandas` and `scipy.sparse`, so it runs on the same Colab/Kaggle image as the rest of the notebook.

In [47]:
# ==============================================================
# 3.1  Content features from schedule metadata + max-pooling model
# ==============================================================
import re
from collections import Counter
from scipy import sparse

_WORD = re.compile(r"[0-9a-zа-яёіїєґ]+")


def _tokens(text, bigrams=True):
    """Lower-case word tokens; optional adjacent bigrams for titles."""
    t = _WORD.findall(str(text).lower())
    if bigrams and len(t) > 1:
        t = t + [f"{a}_{b}" for a, b in zip(t, t[1:])]
    return t


def _l2(X):
    """Row-wise L2 normalisation of a sparse matrix (zero rows stay zero)."""
    n = np.sqrt(np.asarray(X.multiply(X).sum(axis=1)).ravel()).astype(np.float32)
    n[n == 0] = 1.0
    return sparse.diags(1.0 / n).dot(X).astype(np.float32)


def _tfidf(docs, min_df=2):
    """Smoothed tf-idf, sublinear tf, L2-normalised rows."""
    df = Counter()
    for d in docs:
        df.update(set(d))
    vocab = {t: i for i, t in enumerate(sorted(t for t, c in df.items() if c >= min_df))}
    idf = np.array([np.log((1 + len(docs)) / (1 + df[t])) + 1 for t in vocab], dtype=np.float32)
    rows, cols, vals = [], [], []
    for r, d in enumerate(docs):
        for t, n in Counter(d).items():
            j = vocab.get(t)
            if j is not None:
                rows.append(r)
                cols.append(j)
                vals.append((1.0 + np.log(n)) * idf[j])
    X = sparse.csr_matrix((vals, (rows, cols)), shape=(len(docs), len(vocab)), dtype=np.float32)
    return _l2(X)


def _onehot(docs):
    """Multi-hot block from a list of token lists, L2-normalised rows."""
    vocab = sorted({t for d in docs for t in d})
    idx = {t: i for i, t in enumerate(vocab)}
    rows, cols = [], []
    for r, d in enumerate(docs):
        for t in set(d):
            rows.append(r)
            cols.append(idx[t])
    X = sparse.csr_matrix((np.ones(len(rows), np.float32), (rows, cols)),
                          shape=(len(docs), len(vocab)), dtype=np.float32)
    return _l2(X)


def _mode(s):
    vc = s.value_counts(dropna=True)
    return vc.index[0] if len(vc) else np.nan


def _decade(y):
    m = re.match(r"\s*(\d{4})", str(y))
    return f"y{int(m.group(1)) // 10 * 10}" if m else None


def build_item_meta(sched, end_ts):
    """Modal metadata per programme over schedule rows with sched_start < end_ts."""
    s = sched[(sched.tv_show_id != 0) & (sched.sched_start < end_ts)]
    g = s.groupby("tv_show_id", observed=True)
    return pd.DataFrame({
        "title":    g["tv_show_title"].agg(_mode),
        "category": g["tv_show_category"].agg(_mode),
        "g1":       g["tv_show_genre_1"].agg(_mode),
        "g2":       g["tv_show_genre_2"].agg(_mode),
        "g3":       g["tv_show_genre_3"].agg(_mode),
        "year":     g["year_of_production"].agg(_mode),
        "director": g["director"].agg(_mode),
        "actors":   g["actors"].agg(_mode),
    })


def rank_pool(score_row, pool, pop, age_row, k=K):
    """The shared output contract: score DESC -> recency DESC -> popularity DESC -> id ASC."""
    order = np.lexsort((pool, -pop, age_row, -score_row))
    return pool[order[:k]].tolist()


class ContentFeatures:
    """Feature blocks for the union of the candidate pool and the history items."""

    def __init__(self, sched, pool, history, feature_end, min_df=2):
        self.pool = np.asarray(pool, dtype=np.int64)
        hist_ids = {int(i) for i in history["tv_show_id"]}
        self.ids = np.array(sorted(set(self.pool.tolist()) | hist_ids), dtype=np.int64)
        self.iix = {int(i): j for j, i in enumerate(self.ids)}
        meta = build_item_meta(sched, feature_end).reindex(self.ids)
        self.meta = meta

        no_title = int(meta["title"].isna().sum())
        if no_title:
            print(f"  warning: {no_title:,} items have no schedule metadata; they get empty vectors")

        self.title = _tfidf([_tokens(t) for t in meta["title"].astype(object).fillna("").astype(str)],
                            min_df=min_df)

        cat_docs, gen_docs, yr_docs = [], [], []
        for c, a, b, d, y in zip(meta["category"], meta["g1"], meta["g2"],
                                 meta["g3"], meta["year"]):
            cat_docs.append([f"cat_{c}"] if isinstance(c, str) else [])
            gen_docs.append([f"g_{g}" for g in (a, b, d) if isinstance(g, str)])
            dec = _decade(y)
            yr_docs.append([dec] if dec else [])
        self.meta_block = _onehot([c + g + y for c, g, y in zip(cat_docs, gen_docs, yr_docs)])

        cast_docs = []
        for d, a in zip(meta["director"], meta["actors"]):
            text = (d if isinstance(d, str) else "") + " " + (a if isinstance(a, str) else "")
            cast_docs.append(_tokens(text, bigrams=False))
        self.cast = _tfidf(cast_docs, min_df=min_df)

    def matrix(self, w_title=1.0, w_meta=0.7, w_cast=0.4):
        parts = [b * w for b, w in ((self.title, w_title),
                                    (self.meta_block, w_meta),
                                    (self.cast, w_cast)) if w]
        X = parts[0] if len(parts) == 1 else sparse.hstack(parts).tocsr()
        return _l2(X)


class ContentModel:
    """Score pool items by their best-matching item in the user's history.

    pooling="max"      score(i) = max over watched j of  w_j * cos(i, j)
    pooling="centroid" score(i) = cos(i, sum_j w_j * v_j)     (ablation)
    """

    def __init__(self, features, w_title=1.0, w_meta=0.7, w_cast=0.4,
                 half_life=84.0, count="log", pooling="max", max_items=200, k=K):
        self.f = features
        self.w_title, self.w_meta, self.w_cast = w_title, w_meta, w_cast
        self.half_life, self.count, self.pooling = half_life, count, pooling
        self.max_items, self.k = max_items, k

    def fit(self, history, target_start):
        f = self.f
        h = history.copy()
        h["tv_show_id"] = h["tv_show_id"].astype("int64")
        users = sorted({int(u) for u in h["user_id"]})
        self.users = users
        self.uix = {u: i for i, u in enumerate(users)}

        ui = np.array([self.uix[int(u)] for u in h["user_id"]], dtype=np.int64)
        ii = np.array([f.iix[int(i)] for i in h["tv_show_id"]], dtype=np.int64)
        w = (np.log1p(h["n_watch"].to_numpy(np.float32)) if self.count == "log"
             else h["n_watch"].to_numpy(np.float32))
        age_d = (target_start - h["last_ts"].to_numpy()) / 86400.0
        if self.half_life:
            w = w * np.power(0.5, age_d / float(self.half_life)).astype(np.float32)

        X = f.matrix(self.w_title, self.w_meta, self.w_cast)
        Xp = X[[f.iix[int(i)] for i in f.pool]]           # pool x features

        if self.pooling == "centroid":
            P = sparse.csr_matrix((w, (ui, ii)), shape=(len(users), len(f.ids)), dtype=np.float32)
            Pf = _l2(P @ X)                               # users x features
            self.S = np.asarray((Pf @ Xp.T).todense(), dtype=np.float32)
        else:
            Sim = np.asarray((Xp @ X.T).todense(), dtype=np.float32)   # pool x all items
            self.S = np.zeros((len(users), len(f.pool)), dtype=np.float32)
            order = np.argsort(ui, kind="stable")
            ui_s, ii_s, w_s = ui[order], ii[order], w[order]
            starts = np.searchsorted(ui_s, np.arange(len(users) + 1))
            for k in range(len(users)):
                a, b = starts[k], starts[k + 1]
                if b == a:
                    continue
                idx, ww = ii_s[a:b], w_s[a:b]
                if len(idx) > self.max_items:             # keep the most relevant history items
                    keep = np.argpartition(-ww, self.max_items)[:self.max_items]
                    idx, ww = idx[keep], ww[keep]
                self.S[k] = (Sim[:, idx] * ww).max(axis=1)

        pop = h.groupby("tv_show_id").user_id.nunique()
        pop.index = pop.index.astype("int64")
        self.pop = pop.reindex(f.pool).fillna(0).to_numpy(np.float32)

        # recency tie-break: age in days, large for items the user never watched
        self.age = np.full((len(users), len(f.pool)), 1e6, dtype=np.float32)
        pos = {int(i): j for j, i in enumerate(f.pool)}
        m = h["tv_show_id"].isin(f.pool).to_numpy()
        if m.any():
            self.age[ui[m], np.array([pos[int(i)] for i in h.loc[m, "tv_show_id"]], np.int64)] = \
                age_d[m].astype(np.float32)
        return self

    def scorer(self, u):
        ui = self.uix.get(int(u))
        if ui is None:
            return []
        return rank_pool(self.S[ui], self.f.pool, self.pop, self.age[ui], self.k)

### 3.8 Validation grid (A → B) and the selected content model

The cell below builds the feature blocks once, fits each configuration on A, scores B and prints one row per configuration. Nothing here reads B behaviour during fitting: `ContentModel.fit` receives `VAL.history` only; the target is used after the predictions exist, by the harness. The `seconds` column is wall-clock for fit + scoring, so the runtime claim is measured rather than asserted. The last row is the brief's CBF formulation (centroid of category/genre/director/actor embeddings, weighted by raw watch counts); the earlier rows are richer variants that also use title tokens.

In [48]:
# ==============================================================
# 3.2  Fit the content grid on A, score B, keep the validation winner
# ==============================================================
def cold_target_map(problem: Problem) -> dict:
    """Target items with zero views in the fit window (behaviourally cold)."""
    seen = {int(i) for i in problem.history.tv_show_id}
    return {u: [i for i in ids if i not in seen] for u, ids in problem.target.items()}


def evaluate_predictions(problem: Problem, pred: dict, name: str) -> dict:
    """One row of evidence, computed identically for every method."""
    assert problem.target is not None, "evaluation needs a target"
    tgt = problem.target
    warm = sorted(problem.warm_users & set(tgt))

    m_warm, n_warm = map_at_k({u: pred[u] for u in warm}, {u: tgt[u] for u in warm})
    m_all, n_all = map_at_k(pred, tgt)

    hist_by_user = {int(u): {int(i) for i in s}
                    for u, s in problem.history.groupby("user_id").tv_show_id}
    slots = [(int(u), int(i)) for u, ids in pred.items() for i in ids]
    repeat = float(np.mean([i in hist_by_user.get(u, set()) for u, i in slots]))
    coverage = len({i for _, i in slots}) / len(problem.pool)

    cold = cold_target_map(problem)
    den = sum(len(v) for v in cold.values())
    pred_sets = {u: set(ids) for u, ids in pred.items()}
    num = sum(1 for u, ids in cold.items() for i in ids if i in pred_sets[u])
    cold_hit = num / den if den else 0.0

    return dict(method=name,
                map5_warm=round(m_warm, 4), map5_all=round(m_all, 4),
                coverage=round(coverage, 3), repeat_share=round(repeat, 3),
                cold_slot_hit=round(cold_hit, 4),
                fallback_users=len(set(tgt) - problem.warm_users))


def _content_name(cfg):
    return (f"title={cfg['w_title']:g} meta={cfg['w_meta']:g} cast={cfg['w_cast']:g} "
            f"hl={cfg['half_life']} {cfg['pooling']} w={cfg.get('count', 'log')}")


FEATS_VAL = ContentFeatures(sched, VAL.pool, VAL.history, FEATURE_END_VAL)
print(f"content feature rows : {len(FEATS_VAL.ids):,} (pool B + history items)")
print(f"block sizes          : title {FEATS_VAL.title.shape[1]:,} | "
      f"meta {FEATS_VAL.meta_block.shape[1]:,} | cast {FEATS_VAL.cast.shape[1]:,}")

CONTENT_GRID = [
    dict(w_title=1.0, w_meta=0.0, w_cast=0.0, half_life=None, pooling="max"),
    dict(w_title=1.0, w_meta=0.7, w_cast=0.0, half_life=84.0, pooling="max"),
    dict(w_title=1.0, w_meta=0.7, w_cast=0.4, half_life=28.0, pooling="max"),
    dict(w_title=1.0, w_meta=0.7, w_cast=0.4, half_life=84.0, pooling="max"),
    dict(w_title=1.0, w_meta=0.7, w_cast=0.4, half_life=None, pooling="max"),
    dict(w_title=1.0, w_meta=0.7, w_cast=0.4, half_life=84.0, pooling="centroid"),
    # the brief's CBF: centroid of category/genre/director/actor embeddings,
    # weighted by raw watch counts, scored by cosine similarity
    dict(w_title=0.0, w_meta=1.0, w_cast=1.0, half_life=None, pooling="centroid", count="raw"),
]

content_rows = []
CONTENT_BEST, CONTENT_CFG, CONTENT, pred_content = None, None, None, None

for j, cfg in enumerate(CONTENT_GRID):
    t0 = time.time()
    model = ContentModel(FEATS_VAL, k=K, **cfg).fit(VAL.history, VAL.target_start)
    pred = make_predictions(VAL, model.scorer, FALLBACK)
    row = evaluate_predictions(VAL, pred, _content_name(cfg))
    row["seconds"] = round(time.time() - t0, 1)
    content_rows.append(row)
    print(f"{row['method']:52s} MAP {row['map5_warm']:.4f} | cold {row['cold_slot_hit']:.4f} "
          f"| {row['seconds']:5.1f}s")

    # keep only the best model so the grid does not hold six scoring matrices
    if CONTENT_BEST is None or row["map5_warm"] > content_rows[CONTENT_BEST]["map5_warm"]:
        CONTENT_BEST, CONTENT, pred_content = j, model, pred
    else:
        del model, pred
    gc.collect()

content_table = pd.DataFrame(content_rows)
print("\n" + content_table.to_string(index=False))

CONTENT_CFG = CONTENT_GRID[CONTENT_BEST]
print(f"\nselected content configuration: {CONTENT_CFG}")
print(f"validation MAP@5: {content_rows[CONTENT_BEST]['map5_warm']:.4f} (warm) | "
      f"{content_rows[CONTENT_BEST]['map5_all']:.4f} (all)")

content feature rows : 6,975 (pool B + history items)
block sizes          : title 4,673 | meta 86 | cast 5,964
title=1 meta=0 cast=0 hl=None max w=log              MAP 0.1444 | cold 0.0006 |  12.7s
title=1 meta=0.7 cast=0 hl=84.0 max w=log            MAP 0.2656 | cold 0.0018 |  12.9s
title=1 meta=0.7 cast=0.4 hl=28.0 max w=log          MAP 0.2602 | cold 0.0024 |  12.9s
title=1 meta=0.7 cast=0.4 hl=84.0 max w=log          MAP 0.2671 | cold 0.0024 |  13.2s
title=1 meta=0.7 cast=0.4 hl=None max w=log          MAP 0.2672 | cold 0.0024 |  12.8s
title=1 meta=0.7 cast=0.4 hl=84.0 centroid w=log     MAP 0.1395 | cold 0.0012 |   4.1s
title=0 meta=1 cast=1 hl=None centroid w=raw         MAP 0.1406 | cold 0.0018 |   3.6s

                                          method  map5_warm  map5_all  coverage  repeat_share  cold_slot_hit  fallback_users  seconds
         title=1 meta=0 cast=0 hl=None max w=log     0.1444    0.1446     0.180         0.610         0.0006              55     12.7
       tit

**How to read the content table.** `map5_warm` is the full warm population: 3,868 users with A history and exactly five B targets. It is descriptive. The content configuration was chosen from this table among seven configurations, so `map5_warm` is slightly optimistic for content. The number used for selection is the hold-half value `map5_hold` in §4.3 (0.2687). `map5_all` adds the 55 cold users, whose predictions are the shared fallback. `coverage` is the share of pool B that appears anywhere in the top-5 lists. `repeat_share` is the share of recommended slots the user already watched in A. `cold_slot_hit` is the hit rate on target programmes with zero A views; popularity and any purely behavioural method are at 0 there.

The references are popularity 0.2368 and own A-history top-5 0.3263 on the full warm population (0.2320 and 0.3266 on the hold half). If content lands between them, metadata adds reach but not enough signal to beat habit. If it clears 0.3263, content carries information beyond repetition. Either way the number is reported as measured.

## Step 4 — Method 2: personalised PageRank on the co-view graph

### 4.1 The graph

Nodes are the union of pool B and the programmes in the fit-window history, so every candidate can in principle receive mass. Two programmes are connected when the **same user watched both in the fit window** (a co-view edge). The edge weight is the cosine of their viewer sets:

$$W_{ij} = \frac{|U_i \cap U_j|}{\sqrt{|U_i|\,|U_j|}},$$

where $U_i$ is the set of users who watched programme $i$ in A. This is the standard item-item co-view similarity; it is symmetric, in $[0,1]$, and it discounts the fact that popular programmes co-occur with everything.

Two engineering choices keep the walk informative rather than popularity-dominated:

* **Top-$M$ truncation per item** ($M \in \{25, 50, 100\}$ in the grid). A programme that airs 100 times has thousands of weak co-view partners; keeping only its $M$ strongest neighbours removes the "everything is similar to the news" tail and cuts the propagation cost.
* **Mutual edges.** After truncation the graph is symmetrised with $W \leftarrow \max(W, W^\top)$, so an edge survives if either endpoint selected it. The transition matrix is then row-stochastic, $P_{ij}=W_{ij}/\sum_k W_{ik}$.

An item with no co-view partner (a programme nobody else watched, or a test-only item) is a **dangling node**. It gets a self-loop $P_{ii}=1$ so random-walk mass is conserved rather than silently leaking out of the graph.

### 4.2 Personalisation

Each user starts the walk from their own history, with a restart mass **proportional to the number of successful views** of each programme:

$$v_u(j) = \frac{n_{uj}}{\sum_{k \in H_u} n_{uk}}.$$

The vector is a probability distribution over the user's items. No age decay is applied in the shipped configuration: the brief specifies raw watch counts. The sensitivity grid in §4.6 varies only β and the truncation M. Using the same history definition as the content model (same join, same window) keeps the two components comparable.

### 4.3 The walk

Personalised PageRank solves

$$r_u = \alpha P^\top r_u + (1-\alpha) v_u,$$

with damping $\beta=0.7$, exactly as specified. $r_u$ is a distribution over items; the score of pool item $i$ is $r_u(i)$. The walk is **batched over all users**: $R$ is an `items × users` matrix, and each iteration is one sparse-dense product $P^\top R$. This is why the method is practical at this scale — 4,726 users are propagated simultaneously rather than one at a time.

The loop runs **20 power iterations** (the brief's 15–20 range). No tolerance-based early stop is used, so the amount of diffusion is identical for every user and every run. The largest per-user L1 change is still printed as a diagnostic: at $\beta=0.7$ the transient decays as $0.7^{t}$, so after 20 steps the remaining amplitude is $0.7^{20}\approx 0.08\%$ of the initial error — small enough that the ranking is stable, and visible in the `residual` column rather than assumed.

### 4.4 Determinism, repeats and the cold-item limitation

* No randomness: the graph, the transition matrix and the power iteration are deterministic. Ranking uses the same four keys as content: **score DESC → recency DESC → popularity DESC → id ASC**.
* **Repeats are allowed and expected.** The seed distribution is the user's history, so mass is naturally highest near programmes they already watched. This is the correct behaviour for this competition: 53.1 % of the actual B top-5 is a repeat. The method is therefore also a strong test of whether co-view structure adds anything beyond "same again".
* **Behaviourally cold items cannot be reached.** A test-only programme has no A views, hence no co-view edge and zero PPR mass. It is ranked only by the tie-break at the bottom of the list. The `cold_slot_hit` column makes this cost visible; it is a structural limitation of a pure graph method, not a bug. Content is the method that covers this slice.

### 4.5 Selection protocol and cost

The graph is built from A only; every fit uses A and every score uses B. The shipped configuration is the pre-registered one in §4.2 ($M=50$, $\beta=0.7$, 20 iterations, raw-count restart), not the argmax of the sensitivity table. The test graph is built only in §4.8, after the configuration is frozen.

Only the deployed model is kept in memory: each sensitivity row is evaluated, its row recorded, and the model released. The graphs are cached by $M$, because three truncation levels cover all five rows and rebuilding an unchanged graph would waste minutes without changing a number.

Building the co-view matrix is the expensive step: one sparse product of the user-item incidence matrix with its transpose, followed by per-row top-$M$ truncation. Propagation is one sparse-dense product per iteration, vectorised over all users. The dense PageRank state is two `items × users` float32 matrices (current and next iterate), plus the `users × pool` score and age matrices. The implementation is device-agnostic SciPy: it runs unchanged on a GPU runtime but uses that runtime's CPU cores, because there is no CUDA kernel here.

In [49]:
# ==============================================================
# 4.1  Co-view graph + batched personalised PageRank
# ==============================================================
def _top_m_rows(S, top_m):
    """Keep the top_m largest entries of every row of a CSR matrix."""
    S = S.tocsr()
    indptr, indices, data = S.indptr, S.indices, S.data
    out_r, out_c, out_v = [], [], []
    for i in range(S.shape[0]):
        a, b = indptr[i], indptr[i + 1]
        if b == a:
            continue
        idx, val = indices[a:b], data[a:b]
        if len(val) > top_m:
            keep = np.argpartition(-val, top_m - 1)[:top_m]
            idx, val = idx[keep], val[keep]
        out_r.append(np.full(len(idx), i, np.int64))
        out_c.append(idx.astype(np.int64))
        out_v.append(val.astype(np.float32))
    if not out_v:
        return sparse.csr_matrix(S.shape, dtype=np.float32)
    return sparse.csr_matrix(
        (np.concatenate(out_v), (np.concatenate(out_r), np.concatenate(out_c))),
        shape=S.shape, dtype=np.float32)


class CoViewGraph:
    """Item-item cosine co-view graph over union(pool, fit-window history items)."""

    def __init__(self, history, pool, top_m=50):
        self.pool = np.asarray(pool, dtype=np.int64)
        h = history[["user_id", "tv_show_id"]].copy()
        h["tv_show_id"] = h["tv_show_id"].astype("int64")
        self.ids = np.union1d(self.pool, h["tv_show_id"].unique()).astype(np.int64)
        self.iix = {int(i): j for j, i in enumerate(self.ids)}
        self.n_items = len(self.ids)

        users = np.sort(h["user_id"].unique())
        self.uix = {int(u): i for i, u in enumerate(users)}
        ui = np.array([self.uix[int(u)] for u in h["user_id"]], dtype=np.int64)
        ii = np.array([self.iix[int(i)] for i in h["tv_show_id"]], dtype=np.int64)

        B = sparse.csr_matrix((np.ones(len(ui), np.float32), (ui, ii)),
                              shape=(len(users), self.n_items), dtype=np.float32)
        B.sum_duplicates()
        B.data[:] = 1.0                         # binary: one user counts once per programme

        deg = np.asarray(B.sum(axis=0)).ravel().astype(np.float32)
        deg[deg == 0] = 1.0
        Bn = B @ sparse.diags(1.0 / np.sqrt(deg))
        S = (Bn.T @ Bn).tocsr()                 # item-item cosine co-view
        S.setdiag(0.0)
        S.eliminate_zeros()

        W = _top_m_rows(S, top_m)
        W = W.maximum(W.T).tocsr()              # mutual edges only
        W.eliminate_zeros()
        d = np.asarray(W.sum(axis=1)).ravel()
        dangling = d == 0
        P = (sparse.diags(1.0 / np.where(dangling, 1.0, d)) @ W).tocsr()
        if dangling.any():                      # self-loop: never leak random-walk mass
            ix = np.where(dangling)[0]
            P = P + sparse.csr_matrix((np.ones(len(ix), np.float32), (ix, ix)),
                                      shape=P.shape)
        P = P.astype(np.float32).tocsr()
        assert np.allclose(np.asarray(P.sum(axis=1)).ravel(), 1.0, atol=1e-4), \
            "transition matrix is not row-stochastic"
        self.top_m = top_m
        self.W, self.P = W, P
        self.n_edges = int(W.nnz // 2)


class PPRModel:
    """Personalised PageRank from each user's history, batched over all users."""

    def __init__(self, graph, half_life=None, count="raw", alpha=0.7,
                 n_iter=20, tol=None, k=K):
        self.g = graph
        self.half_life, self.count = half_life, count
        self.alpha, self.n_iter, self.tol, self.k = alpha, n_iter, tol, k

    def fit(self, history, target_start):
        g = self.g
        h = history.copy()
        h["tv_show_id"] = h["tv_show_id"].astype("int64")
        users = sorted({int(u) for u in h["user_id"]})
        self.users = users
        self.uix = {u: i for i, u in enumerate(users)}

        ui = np.array([self.uix[int(u)] for u in h["user_id"]], dtype=np.int64)
        ii = np.array([g.iix[int(i)] for i in h["tv_show_id"]], dtype=np.int64)
        w = (np.log1p(h["n_watch"].to_numpy(np.float32)) if self.count == "log"
             else h["n_watch"].to_numpy(np.float32))
        age_d = (target_start - h["last_ts"].to_numpy()) / 86400.0
        if self.half_life:
            w = w * np.power(0.5, age_d / float(self.half_life)).astype(np.float32)

        # teleport vectors: items x users, each column sums to 1
        V = sparse.csr_matrix((w, (ii, ui)), shape=(g.n_items, len(users)), dtype=np.float32)
        col = np.asarray(V.sum(axis=0)).ravel().astype(np.float32)
        col[col == 0] = 1.0
        V = (V @ sparse.diags(1.0 / col)).tocsr()
        Vd = np.asarray(V.todense(), dtype=np.float32)

        # r = beta * P^T r + (1-beta) * v, batched over users.
        # The brief fixes beta = 0.7 and 15-20 iterations, so the default is a
        # fixed 20-step power iteration rather than a tolerance-driven loop.
        Pt = g.P.T.tocsr()                    # transpose once, not per iteration
        R = Vd.copy()
        self.residual = np.inf
        for it in range(self.n_iter):
            R_new = self.alpha * (Pt @ R) + (1.0 - self.alpha) * Vd
            # worst per-user L1 change: a scale-free convergence diagnostic
            self.residual = float(np.abs(R_new - R).sum(axis=0).max())
            R = R_new
            if self.tol is not None and self.residual < self.tol:
                break
        self.n_iter_done = it + 1
        self.R = R

        pool_idx = np.array([g.iix[int(i)] for i in g.pool], dtype=np.int64)
        self.S = np.ascontiguousarray(R[pool_idx].T)          # users x pool

        pop = h.groupby("tv_show_id").user_id.nunique()
        pop.index = pop.index.astype("int64")
        self.pop = pop.reindex(g.pool).fillna(0).to_numpy(np.float32)

        self.age = np.full((len(users), len(g.pool)), 1e6, dtype=np.float32)
        pos = {int(i): j for j, i in enumerate(g.pool)}
        m = h["tv_show_id"].isin(g.pool).to_numpy()
        if m.any():
            self.age[ui[m], np.array([pos[int(i)] for i in h.loc[m, "tv_show_id"]], np.int64)] = \
                age_d[m].astype(np.float32)
        return self

    def scorer(self, u):
        ui = self.uix.get(int(u))
        if ui is None:
            return []
        return rank_pool(self.S[ui], self.g.pool, self.pop, self.age[ui], self.k)


def fit_ppr(problem, top_m=50, **kwargs):
    """Build the graph from the problem's fit window, then run PPR."""
    graph = CoViewGraph(problem.history, problem.pool, top_m=top_m)
    return PPRModel(graph, k=K, **kwargs).fit(problem.history, problem.target_start)

### 4.6 Validation grid (A → B) and the selected graph

The graphs are cached by $M$: the co-view matrix is built once per truncation level, and the $\alpha$ / half-life variants reuse it. That keeps the grid honest (same graph, different walk) and cheap. As in Step 3, the target is used only after the predictions exist.

In [50]:
# ==============================================================
# 4.2  Fit personalised PageRank on A, score B
# ==============================================================
# The brief fixes the walk: cosine-normalised co-view graph, beta = 0.7,
# 15-20 power iterations, restart vector proportional to watch counts.
# PPR_CFG is therefore pre-registered, not selected by peeking at B.
PPR_CFG = dict(top_m=50, alpha=0.70, half_life=None, count="raw", n_iter=20)


def _ppr_name(cfg):
    return (f"topM={cfg['top_m']:>3d} beta={cfg['alpha']:.2f} "
            f"hl={cfg['half_life']} w={cfg['count']} it={cfg['n_iter']}")


graphs = {}


def _fit_ppr_val(cfg):
    if cfg["top_m"] not in graphs:
        graphs[cfg["top_m"]] = CoViewGraph(VAL.history, VAL.pool, top_m=cfg["top_m"])
        g = graphs[cfg["top_m"]]
        print(f"  graph M={cfg['top_m']:>3d}: {g.n_items:,} nodes | "
              f"{g.n_edges:,} undirected edges | P nnz {g.P.nnz:,}")
    kw = {k: v for k, v in cfg.items() if k != "top_m"}
    return PPRModel(graphs[cfg["top_m"]], k=K, **kw).fit(VAL.history, VAL.target_start)


t0 = time.time()
PPR = _fit_ppr_val(PPR_CFG)
pred_ppr = make_predictions(VAL, PPR.scorer, FALLBACK)
ppr_main = evaluate_predictions(VAL, pred_ppr, _ppr_name(PPR_CFG))
print(f"{_ppr_name(PPR_CFG):44s} MAP {ppr_main['map5_warm']:.4f} | "
      f"{PPR.n_iter_done:3d} it | resid {PPR.residual:.1e} | {time.time()-t0:5.1f}s")

# ---- sensitivity only: the deployed model is always the beta = 0.7 row ------
ppr_rows = []
for cfg in [PPR_CFG,
            {**PPR_CFG, "alpha": 0.50},
            {**PPR_CFG, "alpha": 0.85},
            {**PPR_CFG, "top_m": 25},
            {**PPR_CFG, "top_m": 100}]:
    t1 = time.time()
    if cfg is PPR_CFG:
        m, p = PPR, pred_ppr
    else:
        m = _fit_ppr_val(cfg)
        p = make_predictions(VAL, m.scorer, FALLBACK)
    row = evaluate_predictions(VAL, p, _ppr_name(cfg))
    row["iters"] = m.n_iter_done
    row["residual"] = float(f"{m.residual:.2e}")
    row["seconds"] = round(time.time() - t1, 1)
    ppr_rows.append(row)
    if m is not PPR:
        del m, p
    gc.collect()

ppr_table = pd.DataFrame(ppr_rows)
print("\n" + ppr_table.to_string(index=False))
print(f"\ndeployed PPR configuration (pre-registered): {PPR_CFG}")
print("rows after the first are a sensitivity check only; they do not change the shipped model")

  graph M= 50: 6,975 nodes | 77,688 undirected edges | P nnz 159,923
topM= 50 beta=0.70 hl=None w=raw it=20       MAP 0.3362 |  20 it | resid 5.8e-06 |  14.4s
  graph M= 25: 6,975 nodes | 40,823 undirected edges | P nnz 86,193
  graph M=100: 6,975 nodes | 144,975 undirected edges | P nnz 294,497

                                method  map5_warm  map5_all  coverage  repeat_share  cold_slot_hit  fallback_users  iters     residual  seconds
topM= 50 beta=0.70 hl=None w=raw it=20     0.3362    0.3336     0.118         0.961            0.0              55     20 5.790000e-06      0.1
topM= 50 beta=0.50 hl=None w=raw it=20     0.3356    0.3331     0.125         0.961            0.0              55     20 4.680000e-08     14.4
topM= 50 beta=0.85 hl=None w=raw it=20     0.3387    0.3361     0.105         0.959            0.0              55     20 2.820000e-04     14.4
topM= 25 beta=0.70 hl=None w=raw it=20     0.3392    0.3366     0.115         0.959            0.0              55     20 1.03

## Step 4b — Hybrid: multi-component score fusion with weights from a validation search

The hybrid is a score-level fusion of three components. Its weights are chosen on half of the warm validation users and reported on the other half. They are not refit on all users.

* **PPR (personalised PageRank)**: diffusion over the cosine co-view graph (§4.1–4.6). A programme with no A views has no edges and can never receive mass.
* **CBF (content-based filtering)**: the selected content model from Step 3 (§3.7–3.8), using title, category, genre, decade, director and cast.
* **Channel exposure**: for user $u$ and programme $i$, $c(u,i)=\sum_{k} \text{share}(u,k)\cdot n(i,k)$, where $\text{share}(u,k)$ is the user's fraction of A viewing time on channel $k$ and $n(i,k)$ is the number of airings of programme $i$ on channel $k$ in the scored window (weeks 21–30 for validation). The score is high for programmes airing on channels where the user has a large share of viewing.

**Finding weights (§4d.1).** The weights lie on the simplex $w_{\text{PPR}}+w_{\text{CH}}+w_{\text{CBF}}=1$, stored as integers in units of 0.01. Stage 1 scores all 66 lattice points with step 0.10. Stage 2 is coordinate ascent: from the current point, try every move of $d$ units from one component to another, take the best move, and repeat with $d=5$, then 2, then 1 units, up to 25 moves. Each distinct weight vector is scored once. The search uses the tune users only: the even-indexed warm users in sorted order. The hold users (odd-indexed) are only used to report the result. ALS is not used.

**Normalization.** For each user and each component, the score vector is min-max scaled over the candidate pool:

$$\hat s_u(i)=\frac{s_u(i)-\min_j s_u(j)}{\max_j s_u(j)-\min_j s_u(j)}.$$

If a user's vector is constant, the range is set to 1, so that component contributes zero. The fused score is

$$s_u^{hyb}(i)=\sum_k w_k\,\hat s_u^{(k)}(i),\quad k\in\{\text{PPR},\text{channel},\text{CBF}\}.$$

**Candidates, repeats and ties.** Before sorting, the pool is intersected with the programmes that air in the scored window (weeks 21–30 for validation, weeks 31–42 for the test file). Repeats remain eligible. Ranking uses the same total order as every method: **score DESC → last_ts DESC → popularity DESC → tv_show_id ASC**, implemented with `np.lexsort`.

**What it is not.** The fusion acts on scores, per user, after each component has produced its own ranking. It does not fall back to popularity: a cold user receives the shared fallback from `make_predictions`, as in every other method.

In [51]:
# ==============================================================
# 4c.2  Channel exposure component
# ==============================================================
hist_a_all = airings[airings.view_start < VAL_START_S]
user_ch_views = hist_a_all.groupby(['user_id', 'channel_id'])['view_len'].sum().reset_index()
user_ch_total = user_ch_views.groupby('user_id')['view_len'].sum()
user_ch_share = user_ch_views.merge(user_ch_total.rename('total'), on='user_id')
user_ch_share['share'] = user_ch_share['view_len'] / user_ch_share['total']
sched_b = sched_train[(sched_train.sched_start >= VAL_START_S) & (sched_train.sched_start < TEST_START_S) & (sched_train.tv_show_id != 0)]
item_ch_airings = sched_b.groupby(['tv_show_id', 'channel_id']).size().reset_index(name='n_air')
S_CHANNEL = np.zeros((len(CONTENT.users), len(VAL.pool)), dtype=np.float32)
pool_to_idx = {it:i for i,it in enumerate(VAL.pool)}
for idx, u in enumerate(CONTENT.users):
    uch = user_ch_share[user_ch_share['user_id'] == u]
    if len(uch) == 0: continue
    for _, row in uch.iterrows():
        ch = row['channel_id']
        sh = row['share']
        items_ch = item_ch_airings[item_ch_airings['channel_id'] == ch]
        for _, ir in items_ch.iterrows():
            it = ir['tv_show_id']
            if it in pool_to_idx:
                S_CHANNEL[idx, pool_to_idx[it]] += sh * ir['n_air']
print('Channel exposure component added')


Channel exposure component added


In [53]:
# ==============================================================
# 4d.1  Weight search on the simplex: tune half only, hold half for reporting
# ==============================================================
# Weights are integers in units of 1/Q, so every candidate sums to exactly 1.
COMPONENTS = ["ppr", "channel", "cbf"]
Q = 100
GRID_STEP = 10             # 0.10 on the simplex -> 66 candidates
ASCENT_STEPS = (5, 2, 1)   # 0.05, 0.02, 0.01
MAX_MOVES = 25
MIN_GAIN = 1e-9

assert CONTENT.users == PPR.users, "components must share one user order"
assert S_CHANNEL.shape == PPR.S.shape == CONTENT.S.shape
assert CONTENT.f.pool.tolist() == VAL.pool.tolist() == PPR.g.pool.tolist()


def minmax_rows(S):
    S = np.asarray(S, dtype=np.float32)
    lo = S.min(axis=1, keepdims=True)
    hi = S.max(axis=1, keepdims=True)
    rng = hi - lo
    rng[rng == 0] = 1.0
    return (S - lo) / rng


NORM = {"ppr": minmax_rows(PPR.S), "channel": minmax_rows(S_CHANNEL), "cbf": minmax_rows(CONTENT.S)}

VAL_ALLOWED = set(int(i) for i in build_pool(sched, VAL.target_start, TEST_START_S))
POOL, POP, AGE = CONTENT.f.pool, CONTENT.pop, CONTENT.age
ALLOWED = np.isin(POOL, np.fromiter(VAL_ALLOWED, dtype=np.int64, count=len(VAL_ALLOWED)))

# Deterministic split of the warm validation users: weights are chosen on TUNE_USERS only.
WARM_VAL = sorted(VAL.warm_users & set(target_B))
TUNE_USERS = WARM_VAL[0::2]
HOLD_USERS = WARM_VAL[1::2]
assert not set(TUNE_USERS) & set(HOLD_USERS)
print(f"warm validation users {len(WARM_VAL):,} -> tune {len(TUNE_USERS):,} | hold {len(HOLD_USERS):,}")


def weights_from_counts(n):
    return {k: int(c) / Q for k, c in zip(COMPONENTS, n)}


def map_for(weights, users):
    """MAP@5 over warm `users` under fused weights, ranked exactly as make_predictions does."""
    rows = [CONTENT.uix[u] for u in users]
    fused = (weights["ppr"] * NORM["ppr"][rows]
             + weights["channel"] * NORM["channel"][rows]
             + weights["cbf"] * NORM["cbf"][rows])
    pred = {}
    for k, (u, r) in enumerate(zip(users, rows)):
        pred[u] = rank_pool(np.where(ALLOWED, fused[k], -np.inf), POOL, POP, AGE[r], K)
    return map_at_k(pred, {u: target_B[u] for u in users})[0]


def search(users, label):
    cache, trace = {}, []

    def score(n):
        key = tuple(int(c) for c in n)
        if key not in cache:
            cache[key] = map_for(weights_from_counts(key), users)
        return cache[key]

    grid = [np.array([a, b, Q - a - b]) for a in range(0, Q + 1, GRID_STEP)
            for b in range(0, Q + 1 - a, GRID_STEP)]
    best_n = grid[0]
    best_s = score(best_n)
    for n in grid[1:]:
        s = score(n)
        if s > best_s:
            best_n, best_s = n, s
    grid_n, grid_s = best_n.copy(), best_s
    print(f"[{label}] grid: {len(grid)} candidates, best {weights_from_counts(grid_n)} MAP {grid_s:.4f}")

    moves = 0
    for d in ASCENT_STEPS:
        while moves < MAX_MOVES:
            nxt, nxt_s = None, best_s + MIN_GAIN
            for i in range(3):
                for j in range(3):
                    if i == j or best_n[i] < d:
                        continue
                    cand = best_n.copy()
                    cand[i] -= d
                    cand[j] += d
                    s = score(cand)
                    if s > nxt_s:
                        nxt, nxt_s = cand, s
            if nxt is None:
                break
            best_n, best_s = nxt, nxt_s
            moves += 1
            trace.append((d, weights_from_counts(best_n), best_s))
            print(f"[{label}]   step {d:2d}/{Q}  move {moves:2d}: {weights_from_counts(best_n)}  MAP {best_s:.4f}")
    print(f"[{label}] done: {moves} moves, {len(cache)} distinct weight vectors scored")
    return dict(weights=weights_from_counts(best_n), score=best_s,
                grid_weights=weights_from_counts(grid_n), grid_score=grid_s,
                moves=moves, evals=len(cache), trace=trace)


tune = search(TUNE_USERS, "tune")

FIXED = {"ppr": 0.6, "channel": 0.2, "cbf": 0.2}
summary = pd.DataFrame([
    dict(setting="fixed start 0.6/0.2/0.2", weights=FIXED,
         tune=map_for(FIXED, TUNE_USERS), hold=map_for(FIXED, HOLD_USERS)),
    dict(setting="grid best (tune)", weights=tune["grid_weights"],
         tune=tune["grid_score"], hold=map_for(tune["grid_weights"], HOLD_USERS)),
    dict(setting="coordinate ascent (tune) = shipped", weights=tune["weights"],
         tune=tune["score"], hold=map_for(tune["weights"], HOLD_USERS)),
])
summary["weights"] = summary["weights"].map(
    lambda w: f"ppr {w['ppr']:.2f} | ch {w['channel']:.2f} | cbf {w['cbf']:.2f}")
print("\n" + summary.round(4).to_string(index=False))
print("\nthe 'hold' column is the honest estimate; the shipped weights are not refit")

best_weights = tune["weights"]                       # shipped weights: tune-fitted, no refit
HOLD_SCORE_HYBRID = float(summary.iloc[-1]["hold"])
print("\nbest_weights:", best_weights, f"| hold MAP@5 {HOLD_SCORE_HYBRID:.4f}")

warm validation users 3,868 -> tune 1,934 | hold 1,934
[tune] grid: 66 candidates, best {'ppr': 0.5, 'channel': 0.4, 'cbf': 0.1} MAP 0.3489
[tune]   step  5/100  move  1: {'ppr': 0.55, 'channel': 0.35, 'cbf': 0.1}  MAP 0.3490
[tune]   step  5/100  move  2: {'ppr': 0.6, 'channel': 0.35, 'cbf': 0.05}  MAP 0.3491
[tune]   step  1/100  move  3: {'ppr': 0.59, 'channel': 0.36, 'cbf': 0.05}  MAP 0.3491
[tune] done: 3 moves, 90 distinct weight vectors scored

                           setting                       weights   tune   hold
           fixed start 0.6/0.2/0.2 ppr 0.60 | ch 0.20 | cbf 0.20 0.3449 0.3442
                  grid best (tune) ppr 0.50 | ch 0.40 | cbf 0.10 0.3489 0.3489
coordinate ascent (tune) = shipped ppr 0.59 | ch 0.36 | cbf 0.05 0.3491 0.3489

the 'hold' column is the honest estimate; the shipped weights are not refit

best_weights: {'ppr': 0.59, 'channel': 0.36, 'cbf': 0.05} | hold MAP@5 0.3489


In [55]:
# ==============================================================
# 4b.1  Hybrid: fusion with the tune-fitted weights, scored on the hold half
# ==============================================================
HYBRID_WEIGHTS = dict(best_weights)


def schedule_ids(sched, start, end):
    return set(int(i) for i in build_pool(sched, start, end))


def minmax_rows(S):
    S = np.asarray(S, dtype=np.float32)
    lo = S.min(axis=1, keepdims=True)
    hi = S.max(axis=1, keepdims=True)
    rng = hi - lo
    rng[rng == 0] = 1.0
    return (S - lo) / rng


def fuse_scores_multi(components_dict, weights_dict):
    ref = list(components_dict.keys())[0]
    n_users = components_dict[ref].shape[0]
    n_pool = components_dict[ref].shape[1]
    fused = np.zeros((n_users, n_pool), dtype=np.float32)
    for name, w in weights_dict.items():
        if name in components_dict and w > 0:
            fused += float(w) * minmax_rows(components_dict[name])
    return fused


def hybrid_scorer_multi(fused, allowed_ids):
    uix, pool = CONTENT.uix, CONTENT.f.pool
    pop, age = CONTENT.pop, CONTENT.age
    allowed = np.isin(pool, np.fromiter(allowed_ids, dtype=np.int64, count=len(allowed_ids)))

    def scorer(u):
        ui = uix.get(int(u))
        if ui is None:
            return []
        return rank_pool(np.where(allowed, fused[ui], -np.inf), pool, pop, age[ui], K)
    return scorer


VAL_ALLOWED = schedule_ids(sched, VAL.target_start, TEST_START_S)
_comps = {'ppr': PPR.S, 'cbf': CONTENT.S, 'channel': S_CHANNEL}
S_HYBRID = fuse_scores_multi(_comps, HYBRID_WEIGHTS)
pred_hybrid = make_predictions(VAL, hybrid_scorer_multi(S_HYBRID, VAL_ALLOWED), FALLBACK)

hold_hybrid = map_at_k({u: pred_hybrid[u] for u in HOLD_USERS}, {u: target_B[u] for u in HOLD_USERS})[0]
print(f"hybrid weights (tune-fitted): {HYBRID_WEIGHTS}")
print(f"hybrid hold MAP@5: {hold_hybrid:.4f}")

hybrid weights (tune-fitted): {'ppr': 0.59, 'channel': 0.36, 'cbf': 0.05}
hybrid hold MAP@5: 0.3489


### 4.7 Head-to-head on the identical split

One table, one harness, one fallback, one pool, one target. `map5_hold` (the hold half of the 3,868 warm validation users, 1,934 users) is the selection column. `map5_warm` and `map5_all` are descriptive: they include the tune half, so the hybrid's values there are in-sample.

In [56]:
# ==============================================================
# 4.3  Head-to-head: every method on the hold half (used for selection)
# ==============================================================
assert CONTENT.users == PPR.users, "methods were fitted on different users"
assert CONTENT.f.pool.tolist() == VAL.pool.tolist() == PPR.g.pool.tolist()
print(f"same fit population for both methods: {len(CONTENT.users):,} users | "
      f"same pool: {len(VAL.pool):,} programmes")

CANDIDATES_VAL = [
    ("popularity (reference)",           pred_pop,     "none"),
    ("own A-history top-5 (reference)",  pred_own,     "none"),
    ("content-based (method 1)",         pred_content, "none"),
    ("personalised PageRank (method 2)", pred_ppr,     "none"),
    ("hybrid multi-component",           pred_hybrid,  "tune half"),
]

rows = []
for name, pred, in_sample in CANDIDATES_VAL:
    row = evaluate_predictions(VAL, pred, name)
    row["map5_hold"] = round(map_at_k({u: pred[u] for u in HOLD_USERS},
                                      {u: target_B[u] for u in HOLD_USERS})[0], 4)
    row["in_sample_users"] = in_sample
    rows.append(row)

comparison = pd.DataFrame(rows)[
    ["method", "map5_hold", "map5_warm", "map5_all", "coverage", "repeat_share",
     "cold_slot_hit", "fallback_users", "in_sample_users"]
]
print(comparison.to_string(index=False))
print("\nmap5_hold is the selection number; map5_warm/map5_all include the tune half for the hybrid")

for label, pred in [("popularity", pred_pop), ("own", pred_own),
                    ("content", pred_content), ("PPR", pred_ppr),
                    ("hybrid", pred_hybrid)]:
    assert set(pred) == set(VAL.users), label
    assert all(len(ids) == K for ids in pred.values()), label
    assert all(set(ids) <= VAL._pool_set for ids in pred.values()), label
print(f"\noutput contract OK: {len(VAL.users):,} users x exactly {K} ids, all inside pool B")

same fit population for both methods: 4,726 users | same pool: 6,376 programmes
                          method  map5_hold  map5_warm  map5_all  coverage  repeat_share  cold_slot_hit  fallback_users in_sample_users
          popularity (reference)     0.2320     0.2368    0.2357     0.001         0.627         0.0000              55            none
 own A-history top-5 (reference)     0.3266     0.3263    0.3240     0.132         0.961         0.0000              55            none
        content-based (method 1)     0.2687     0.2672    0.2657     0.113         0.634         0.0024              55            none
personalised PageRank (method 2)     0.3357     0.3362    0.3336     0.118         0.961         0.0000              55            none
          hybrid multi-component     0.3489     0.3490    0.3463     0.085         0.909         0.0006              55       tune half

map5_hold is the selection number; map5_warm/map5_all include the tune half for the hybrid

output cont

### 4.8 Freezing the configurations and scoring the test pool

This is the point where the test schedule is used. Neither method sees test behaviour — there is none — and neither configuration is re-tuned: the exact winning dictionaries from validation are passed to fresh fits on A ∪ B.

The test fallback is rebuilt on the same rule as validation: global popularity over A ∪ B restricted to pool T, ties by last view start, then id. It is one list for every method and for the users with no history, so the comparison on test is again like-for-like.

The hybrid uses the tune-fitted weights 0.59 / 0.36 / 0.05 and is not refit. The printed line `warm 4,834 | fallback -3,574` counts users with any history (4,834) against the 1,260 submission users, so it is not a count of submission users. Of the 1,260 submission users, 1,255 have history and 5 use the fallback (§1.5). The negative number is a display error in the cell; `make_predictions` tests membership in the history set, so the predictions are unaffected.

In [57]:
# ==============================================================
# 4.4  Refit the frozen configurations on A ∪ B and score the test pool
# ==============================================================
# No test-window behaviour exists or is read here. The content and PPR configurations and the
# hybrid weights are the ones chosen on validation; nothing is re-tuned on the test pool.
for _name in ["graphs", "CONTENT", "PPR", "FEATS_VAL", "S_HYBRID", "NORM"]:
    if _name in globals():
        del globals()[_name]
gc.collect()

pop_AB = popularity_scores(hist_AB, P_T)
pop_AB_tie = hist_AB.groupby("tv_show_id").last_ts.max()
FALLBACK_TEST = rank_items(pop_AB, K, tiebreak=pop_AB_tie.to_dict())

FEATS_TEST = ContentFeatures(sched, TEST.pool, TEST.history, FEATURE_END_TEST)
CONTENT_TEST = ContentModel(FEATS_TEST, k=K, **CONTENT_CFG).fit(TEST.history, TEST.target_start)
PPR_TEST = fit_ppr(TEST, **PPR_CFG)

assert CONTENT_TEST.users == PPR_TEST.users, "test components must share one user order"
assert PPR_TEST.g.pool.tolist() == TEST.pool.tolist(), "PPR_TEST pool is not TEST.pool"
assert CONTENT_TEST.f.pool.tolist() == TEST.pool.tolist(), "CONTENT_TEST pool is not TEST.pool"

pred_content_test = make_predictions(TEST, CONTENT_TEST.scorer, FALLBACK_TEST)
pred_ppr_test = make_predictions(TEST, PPR_TEST.scorer, FALLBACK_TEST)

TEST_ALLOWED = set(int(i) for i in build_pool(sched, TEST_START_S, 2 ** 40))

# Channel exposure for the test refit: the same definition as 4c.2, with A ∪ B history
# and the weeks 31–42 schedule for the airings.
user_ch_views_t = (airings[airings.view_start < TEST_START_S]
                   .groupby(["user_id", "channel_id"])["view_len"].sum().reset_index())
user_ch_total_t = user_ch_views_t.groupby("user_id")["view_len"].sum()
user_ch_share_t = user_ch_views_t.merge(user_ch_total_t.rename("total"), on="user_id")
user_ch_share_t["share"] = user_ch_share_t["view_len"] / user_ch_share_t["total"]

sched_t_air = sched[(sched.sched_start >= TEST_START_S) & (sched.sched_start < 2 ** 40)
                    & (sched.tv_show_id != 0)]
item_ch_airings_t = sched_t_air.groupby(["tv_show_id", "channel_id"]).size().reset_index(name="n_air")
airings_by_ch = {int(c): g for c, g in item_ch_airings_t.groupby("channel_id")}

test_row = {int(u): i for i, u in enumerate(PPR_TEST.users)}
test_col = {int(i): j for j, i in enumerate(TEST.pool)}
S_CHANNEL_TEST = np.zeros((len(PPR_TEST.users), len(TEST.pool)), dtype=np.float32)
for u, c, sh in zip(user_ch_share_t.user_id, user_ch_share_t.channel_id, user_ch_share_t.share):
    i, g = test_row.get(int(u)), airings_by_ch.get(int(c))
    if i is None or g is None:
        continue
    for it, n_air in zip(g.tv_show_id, g.n_air):
        j = test_col.get(int(it))
        if j is not None:
            S_CHANNEL_TEST[i, j] += float(sh) * float(n_air)

assert set(HYBRID_WEIGHTS) == {"ppr", "channel", "cbf"}, HYBRID_WEIGHTS
assert S_CHANNEL_TEST.shape == PPR_TEST.S.shape == CONTENT_TEST.S.shape
HYBRID_WEIGHTS_T = {k: float(v) for k, v in HYBRID_WEIGHTS.items() if v > 0}
_comps_t = {"ppr": PPR_TEST.S, "channel": S_CHANNEL_TEST, "cbf": CONTENT_TEST.S}
S_HYBRID_TEST = fuse_scores_multi(_comps_t, HYBRID_WEIGHTS_T)
print(f"test hybrid weights (tune-fitted, not refit): {HYBRID_WEIGHTS_T}")

uix_t, pool_t = CONTENT_TEST.uix, CONTENT_TEST.f.pool
pop_t, age_t = CONTENT_TEST.pop, CONTENT_TEST.age
allowed_t_arr = np.isin(pool_t, np.fromiter(TEST_ALLOWED, dtype=np.int64, count=len(TEST_ALLOWED)))


def scorer_t(u):
    ui = uix_t.get(int(u))
    if ui is None:
        return []
    return rank_pool(np.where(allowed_t_arr, S_HYBRID_TEST[ui], -np.inf), pool_t, pop_t, age_t[ui], K)


pred_hybrid_test = make_predictions(TEST, scorer_t, FALLBACK_TEST)

print(f"test pool              : {len(TEST.pool):,} programmes")
print(f"submission users       : {len(TEST.users):,} | warm {len(TEST.warm_users):,} | "
      f"fallback {len(TEST.users) - len(TEST.warm_users):,}")
print(f"test fallback (pop A+B): {FALLBACK_TEST}")

for label, pred in [("content", pred_content_test), ("PPR", pred_ppr_test),
                    ("hybrid", pred_hybrid_test)]:
    ids = [i for v in pred.values() for i in v]
    assert set(pred) == set(TEST.users), label
    assert all(len(v) == K and set(v) <= TEST._pool_set for v in pred.values()), label
    top = pd.Series(ids).value_counts()
    print(f"  {label:8s}: {len(pred):,} users | {len(set(ids)):,} distinct ids | "
          f"most frequent {int(top.index[0])} ({top.iloc[0]:,} slots)")

del S_HYBRID_TEST, CONTENT_TEST, PPR_TEST, S_CHANNEL_TEST
gc.collect()
print("\ntest predictions are held in memory; §5.1 applies the hold-out policy and writes submission.csv")

test hybrid weights (tune-fitted, not refit): {'ppr': 0.59, 'channel': 0.36, 'cbf': 0.05}
test pool              : 6,646 programmes
submission users       : 1,260 | warm 4,834 | fallback -3,574
test fallback (pop A+B): [2400480, 240081, 2400467, 2400508, 200344]
  content : 1,260 users | 309 distinct ids | most frequent 2400480 (939 slots)
  PPR     : 1,260 users | 269 distinct ids | most frequent 2400480 (948 slots)
  hybrid  : 1,260 users | 235 distinct ids | most frequent 2400480 (927 slots)

test predictions are held in memory; §5.1 applies the hold-out policy and writes submission.csv


## Step 3–4 — what the validation tables support

The selection column is `map5_hold` (§4.3). It is computed on the hold half of the warm validation users, which has 1,934 users.

| Method | map5_hold | map5_warm (descriptive) | coverage | repeat_share | cold_slot_hit |
|---|---|---|---|---|---|
| popularity (reference) | 0.2320 | 0.2368 | 0.001 | 0.627 | 0.0000 |
| own A-history top-5 (reference) | 0.3266 | 0.3263 | 0.132 | 0.961 | 0.0000 |
| content (method 1) | 0.2687 | 0.2672 | 0.113 | 0.634 | 0.0024 |
| PPR (method 2) | 0.3357 | 0.3362 | 0.118 | 0.961 | 0.0000 |
| hybrid (PPR + channel + CBF) | 0.3489 | 0.3490 | 0.085 | 0.909 | 0.0006 |

**What these numbers support.**

* Every method beats popularity on the hold half. The hybrid beats PPR by 0.3489 − 0.3357 = 0.0132 and own history by 0.3489 − 0.3266 = 0.0223.
* Content alone beats popularity (0.2687 − 0.2320 = +0.0367) but not own history (0.2687 − 0.3266 = −0.0579).
* The coordinate ascent did not improve the held-out result. The grid's best point (PPR 0.50, channel 0.40, CBF 0.10) scores 0.3489 on hold, and the ascent's point (0.59 / 0.36 / 0.05) scores 0.3489. The ascent moved the tune score from 0.3489 to 0.3491 and nothing on hold.
* The hold half has 1,934 users, and 83.6 % of users have a tied 5th frequency value. Differences of a few thousandths are inside label noise. No confidence interval is computed in this notebook.

**Why the methods are complementary.** Content reaches programmes with no co-view history, PPR uses co-view patterns, and channel exposure adds the habit of channel choice. The hybrid combines them with the weights from §4b.

**Policy.** §5.1 selects the candidate with the highest hold-out MAP@5. Individual prediction files are kept for auditability. Popularity and own history are references only and are never submitted.

## Step 5 — Final strategy and `submission.csv`

### 5.1 The decision rule, fixed before the numbers are read

Three candidates are on the table: **content-based (method 1)**, **personalised PageRank (method 2)**, and the **multi-component hybrid** (PPR + channel exposure + CBF), fused with the weights chosen on the tune half.

The rule is:

1. Deploy the candidate with the highest **hold-out MAP@5** (`map5_hold`) on the hold half of the warm validation users (1,934 users).
2. Ties go to the hybrid.
3. **Popularity** and **own history** are reference rows. They are scored and compared here, and they are never written to a submission file.

Selection uses only the hold half, because the hybrid's weights were chosen on the tune half. In the run whose outputs are saved here, the hybrid wins with 0.3489 against 0.3357 for PPR.

### 5.2 Every candidate is saved separately

The cell writes four files:

| File | Meaning |
|---|---|
| `submission.csv` | the chosen candidate, the one to upload |
| `submission_content.csv` | method 1 predictions for all 1,260 users |
| `submission_ppr.csv` | method 2 predictions for all 1,260 users |
| `submission_hybrid.csv` | the multi-component hybrid predictions for all 1,260 users |

### 5.3 The file contract

The organiser's template is `user_id,tv_show_id`, one row per submission user, and the five ids are **space-separated inside a single string**. Every file preserves the template's row order and `uint64` ids, and is re-read and checked: two columns, 1,260 rows, exactly five distinct ids per row, every id inside pool T. `submission.csv` is written from the winning method's predictions and verified the same way. In this run it has 235 distinct ids.

In [60]:
# ==============================================================
# 5.1  Apply the hold-out policy, then write the submission files
# ==============================================================
t0 = time.time()

NAME_CONTENT = "content-based (method 1)"
NAME_PPR     = "personalised PageRank (method 2)"
NAME_HYBRID  = "hybrid multi-component"
NAME_POP     = "popularity (reference)"

# Selection uses the hold half only; the hybrid's weights were fitted on the tune half.
hold_map = comparison.set_index("method")["map5_hold"].to_dict()
CANDIDATE_NAMES = [NAME_CONTENT, NAME_PPR, NAME_HYBRID]
assert all(n in hold_map for n in CANDIDATE_NAMES + [NAME_POP]), "comparison is missing a row"

FINAL_NAME = max(CANDIDATE_NAMES, key=lambda n: (hold_map[n], n == NAME_HYBRID))
RUNNER_UP = max((n for n in CANDIDATE_NAMES if n != FINAL_NAME), key=lambda n: hold_map[n])

HYBRID_WEIGHTS_RECORD = {k: float(v) for k, v in HYBRID_WEIGHTS.items() if v > 0}
FINAL_POLICY = dict(
    policy=FINAL_NAME,
    runner_up=RUNNER_UP,
    selection_split=f"hold half, {len(HOLD_USERS):,} warm validation users",
    rule="highest hold-out MAP@5 among content, PPR and the hybrid; tie -> hybrid",
    hold_map5=round(float(hold_map[FINAL_NAME]), 4),
    gap_vs_runner_up=round(float(hold_map[FINAL_NAME] - hold_map[RUNNER_UP]), 4),
    gap_vs_popularity=round(float(hold_map[FINAL_NAME] - hold_map[NAME_POP]), 4),
)

print("validation MAP@5 on the hold half; identical harness, fallback and target for every row")
print(comparison[["method", "map5_hold", "map5_warm", "coverage",
                  "repeat_share", "cold_slot_hit", "in_sample_users"]].to_string(index=False))
print("\nmap5_warm includes the tune half and is descriptive only; it is not used for selection")
print("popularity and own-history are reference rows: compared here, never submitted")
print(f"\nchosen method : {FINAL_NAME}")
print(f"rule          : {FINAL_POLICY['rule']}")
print(f"selection     : {FINAL_POLICY['selection_split']}")
print(f"hold MAP@5    : {FINAL_POLICY['hold_map5']:.4f} | "
      f"{FINAL_POLICY['gap_vs_runner_up']:+.4f} vs {RUNNER_UP}")
print(f"vs popularity : {FINAL_POLICY['gap_vs_popularity']:+.4f} MAP@5 (hold)")

SUBMISSION_DIR = Path(os.environ.get("SWEETTV_SUBMISSION_DIR", "."))
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)
SUB_PATH = SUBMISSION_DIR / "submission.csv"

template = pd.read_csv(DATA / "submission.csv", dtype={"user_id": "uint64"})
assert len(template) == len(TEST.users), \
    f"template has {len(template)} rows but {len(TEST.users)} users were scored"
assert set(int(u) for u in template.user_id) == set(int(u) for u in TEST.users), \
    "template users and scored users differ"


def write_and_verify(pred, path):
    """Write one prediction set in the template's exact shape, then re-read and assert it."""
    id_string = {int(u): " ".join(str(int(i)) for i in ids) for u, ids in pred.items()}
    assert set(id_string) == set(int(u) for u in template.user_id), \
        f"{path.name}: users do not match the template"
    frame = pd.DataFrame({
        "user_id": template.user_id,
        "tv_show_id": [id_string[int(u)] for u in template.user_id],
    })
    frame.to_csv(path, index=False)

    chk = pd.read_csv(path, dtype={"user_id": "uint64", "tv_show_id": "string"})
    toks = chk.tv_show_id.str.split()
    assert list(chk.columns) == ["user_id", "tv_show_id"]
    assert len(chk) == len(template)
    assert chk.user_id.to_numpy().tolist() == template.user_id.to_numpy().tolist()
    assert chk.tv_show_id.notna().all()
    assert toks.str.len().eq(K).all(), f"{path.name}: a row does not have five ids"
    assert toks.map(lambda v: len(set(v)) == K).all(), f"{path.name}: duplicate id in a row"
    flat = toks.explode()
    assert flat.notna().all()
    flat = flat.astype("int64")
    assert flat.isin(TEST.pool).all(), f"{path.name}: id outside pool T"
    return frame, chk, flat


CANDIDATES = {NAME_CONTENT: pred_content_test, NAME_PPR: pred_ppr_test,
              NAME_HYBRID: pred_hybrid_test}
SLUG = {NAME_CONTENT: "content", NAME_PPR: "ppr", NAME_HYBRID: "hybrid"}
CHECKS, FLATS = {}, {}
for label, pred in CANDIDATES.items():
    path = SUBMISSION_DIR / f"submission_{SLUG[label]}.csv"
    _, CHECKS[label], FLATS[label] = write_and_verify(pred, path)
    print(f"wrote {path.resolve()} ({path.stat().st_size/1024:.1f} KB)")

final_frame, final_check, final_flat = write_and_verify(CANDIDATES[FINAL_NAME], SUB_PATH)
print(f"wrote {SUB_PATH.resolve()} <- {FINAL_NAME} ({SUB_PATH.stat().st_size/1024:.1f} KB)")

print("\nfile contract (each file re-read and asserted)")
for label, chk, flat in [
    ("submission_content.csv", CHECKS[NAME_CONTENT], FLATS[NAME_CONTENT]),
    ("submission_ppr.csv",     CHECKS[NAME_PPR],     FLATS[NAME_PPR]),
    ("submission_hybrid.csv",  CHECKS[NAME_HYBRID],  FLATS[NAME_HYBRID]),
    ("submission.csv (chosen)", final_check,          final_flat),
]:
    print(f"  {label:26s} rows {len(chk):,} | {flat.nunique():5,} distinct ids | "
          f"5 distinct/row OK | inside pool T OK")

CMP_PATH = save_df(comparison, "validation_comparison")
print(f"\nvalidation comparison saved to {CMP_PATH}")

run_ts = pd.Timestamp.now(tz="UTC").strftime("%Y%m%d-%H%M%S")
manifest = dict(
    timestamp_utc=run_ts,
    chosen_method=FINAL_NAME,
    runner_up=RUNNER_UP,
    rule=FINAL_POLICY["rule"],
    selection_split=FINAL_POLICY["selection_split"],
    hold_map5={
        "content": round(float(hold_map[NAME_CONTENT]), 4),
        "ppr": round(float(hold_map[NAME_PPR]), 4),
        "hybrid_multi_component": round(float(hold_map[NAME_HYBRID]), 4),
        "popularity_reference": round(float(hold_map[NAME_POP]), 4),
    },
    fusion=dict(weights=HYBRID_WEIGHTS_RECORD,
                weights_fitted_on=f"tune half, {len(TUNE_USERS):,} warm users; not refit",
                normalisation="per-user min-max over programmes airing in the scored window"),
    content_config=CONTENT_CFG,
    ppr_config=PPR_CFG,
    files=["submission.csv", "submission_content.csv",
           "submission_ppr.csv", "submission_hybrid.csv"],
    rows=int(len(final_check)),
    distinct_ids_chosen=int(final_flat.nunique()),
    pool_T=int(len(TEST.pool)),
    test_fallback=[int(i) for i in FALLBACK_TEST],
    baseline_submitted=False,
    python=platform.python_version(),
    pandas=pd.__version__,
)

run_dir = SUBMISSION_DIR / "submission_runs"
run_dir.mkdir(exist_ok=True)
snap = run_dir / f"submission_{run_ts}.csv"
n = 1
while snap.exists():
    n += 1
    snap = run_dir / f"submission_{run_ts}_{n}.csv"
final_frame.to_csv(snap, index=False)
with open(run_dir / "manifest.jsonl", "a", encoding="utf-8") as fh:
    fh.write(json.dumps(manifest, ensure_ascii=False) + "\n")
print(f"archived run  : {snap.resolve()}")
print(f"manifest      : {(run_dir / 'manifest.jsonl').resolve()}")

print("\nrun record:")
print(json.dumps(manifest, indent=2, ensure_ascii=False))
print(f"\nStep 5 runtime: {time.time()-t0:.1f} s")

validation MAP@5 on the hold half; identical harness, fallback and target for every row
                          method  map5_hold  map5_warm  coverage  repeat_share  cold_slot_hit in_sample_users
          popularity (reference)     0.2320     0.2368     0.001         0.627         0.0000            none
 own A-history top-5 (reference)     0.3266     0.3263     0.132         0.961         0.0000            none
        content-based (method 1)     0.2687     0.2672     0.113         0.634         0.0024            none
personalised PageRank (method 2)     0.3357     0.3362     0.118         0.961         0.0000            none
          hybrid multi-component     0.3489     0.3490     0.085         0.909         0.0006       tune half

map5_warm includes the tune half and is descriptive only; it is not used for selection
popularity and own-history are reference rows: compared here, never submitted

chosen method : hybrid multi-component
rule          : highest hold-out MAP@5 among c

**What this cell prints.** The `comparison` table holds the five rows used for the decision. Selection uses `map5_hold`, the same metric (`map_at_k`) on the hold half of the warm validation users. `map5_warm` is shown for description and includes the tune half. Popularity and own history are reference rows and are never submitted. The cell writes the chosen candidate to `submission.csv`, writes the three candidate sets to their own files, and re-reads each file to check the template row order, the `uint64` ids, five distinct ids per row, and ids inside pool T.

## Step 6 — Final strategy: multi-component hybrid

The submitted strategy is a score-level fusion of PPR, channel exposure and CBF, with weights PPR 0.59, channel 0.36 and CBF 0.05 from the search in §4b and §4d.1. The weights were chosen on the tune half of the warm validation users and are not refit. The hold-out MAP@5 of 0.3489 is the validation estimate.

### Components

| Component | Role | Notes |
|---|---|---|
| Personalised PageRank (PPR) | co-view diffusion | graph-based, strong on repeat and co-view patterns |
| Content-based filtering (CBF) | metadata similarity | weight 0.05; reaches programmes with little co-view history |
| Channel exposure | viewing habit by channel | the user's channel shares times the airings of each programme on those channels |

### Fusion and test refit

- Per-user min-max normalisation over the candidate pool for each component, then a weighted sum.
- Candidates are restricted to programmes airing in the scored window (weeks 21–30 for validation, weeks 31–42 for test).
- Ties are broken by last_ts DESC, then popularity DESC, then tv_show_id ASC.
- The top five comes from `make_predictions`, with the shared fallback for users without history.
- For the test, PPR and CBF are refit on A ∪ B with the frozen configurations, the channel component is rebuilt from A ∪ B views and weeks 31–42 airings, and the same weights are applied. No test-window viewing is used.

## Step 7 — Recommendation examples: two users

Two users are drawn from the validation population, because weeks 21–30 are the only window where "what the user actually watched" can be inspected without touching the test period. The models that produce their lists were fitted on weeks 11–20 only, so nothing from the target window enters the recommendation.

The cell below:

* picks one **heavy** and one **light** viewer by distinct programmes in A, and lets you override the choice with `EXAMPLE_USERS`;
* accepts a **cold** user as well — the available cold validation ids are printed so they can be copied into `EXAMPLE_USERS`;
* looks up title and category from the schedule metadata, never from the view log;
* prints, per user, the A history, the actual B viewing, and the top-5 of every method in rank order;
* repeats the five ids on one line per method, ready to paste into the report;
* marks whether a recommended id was already in A (familiar) and whether it appears in the B target (`hit #k`).

Every list has exactly five ids because `make_predictions` appends the shared popularity fallback when a user has no history. For a cold user all five lists are identical by construction — that is the fallback, not a method result.

The printed hybrid label reads `hybrid 0.7*PPR + 0.3*CBF`. That is a leftover label in the code and does not describe the shipped hybrid, which uses PPR 0.59, channel exposure 0.36 and CBF 0.05 (§4b, §5.1). In this run the automatic choice was a heavy viewer (262 programmes in A) and a light viewer (one programme, two airings in A).

In the printed lists, `in_B_target` and `hit #k` mean that the programme was watched in weeks 21–30 at all; `hit #k` gives its rank in the user's B list. The five programmes that define MAP@5 are the `scored_target` marks in the B table. The summary line 'against the actual B target' counts B-watched programmes in the same way, so it is not the MAP@5 target; use the `scored_target` column for MAP@5.

In [61]:
# ==============================================================
# 7.1  Two users: each method's top-5 beside what they actually watched
# ==============================================================
# Override the automatic choice by listing two user ids as integers.
# Ids are uint64 and exceed 2**53, so never type them as floats.
EXAMPLE_USERS = None          # e.g. [8387147770138767246, 8377619604347126107]

A_SHOWS = {int(u): int(n) for u, n in hist_A.groupby("user_id").tv_show_id.nunique().items()}
A_VIEWS = {int(u): int(n) for u, n in hist_A.groupby("user_id").n_watch.sum().items()}

TARGET_USERS      = sorted(int(u) for u in target_B)
WARM_TARGET_USERS = [u for u in TARGET_USERS if u in VAL.warm_users]
COLD_TARGET_USERS = [u for u in TARGET_USERS if u not in VAL.warm_users]
assert WARM_TARGET_USERS, "no warm validation user has a complete five-item target"

if EXAMPLE_USERS is None:
    heavy = max(WARM_TARGET_USERS, key=lambda u: A_SHOWS.get(u, 0))
    light = min(WARM_TARGET_USERS, key=lambda u: A_SHOWS.get(u, 10**9))
    if light == heavy:                       # pathological: all users have the same history size
        light = next(u for u in WARM_TARGET_USERS if u != heavy)
    EXAMPLE_USERS = [heavy, light]

EXAMPLE_USERS = [int(u) for u in EXAMPLE_USERS]
assert len(set(EXAMPLE_USERS)) == 2, "pick two distinct users"
for u in EXAMPLE_USERS:
    assert u in target_B, f"{u} has no complete five-item target in weeks 21-30"

print("example users")
for u in EXAMPLE_USERS:
    kind = "cold (no A history)" if u not in VAL.warm_users else "warm"
    print(f"  {u}  {kind:18s} | A programmes {A_SHOWS.get(u, 0):3d} | "
          f"A airings {A_VIEWS.get(u, 0):4d} | B target {len(target_B[u])}")
if COLD_TARGET_USERS:
    print(f"  cold validation ids available for an example: {COLD_TARGET_USERS[:8]} ...")
    print(f"  set EXAMPLE_USERS = [{EXAMPLE_USERS[0]}, {COLD_TARGET_USERS[0]}] to study one")
print(f"  shared fallback (identical for every method): {FALLBACK}")
print(f"  output contract: every list below has exactly {K} ids; a short list is padded from FALLBACK")

ITEM_META = build_item_meta(sched, FEATURE_END_TEST)


def item_label(i):
    """Title and category of a programme, modal over its schedule rows."""
    i = int(i)
    try:
        row = ITEM_META.loc[i]
    except KeyError:
        return "(not in schedule metadata)", "(unknown)"
    title = row["title"] if isinstance(row["title"], str) and row["title"] else "(untitled)"
    cat   = row["category"] if isinstance(row["category"], str) and row["category"] else "(uncategorised)"
    return title, cat


METHOD_PREDS = [
    ("content",    "content-based (method 1)",        pred_content),
    ("ppr",        "personalised PageRank (method 2)", pred_ppr),
    ("hybrid",     "hybrid 0.7*PPR + 0.3*CBF",        pred_hybrid),
    ("popularity", "popularity (reference)",          pred_pop),
    ("own",        "own A-history top-5 (reference)", pred_own),
]


def show_user(u):
    u = int(u)
    h = hist_A[hist_A.user_id == u].copy()
    h = h.sort_values(["n_watch", "last_ts", "tv_show_id"],
                      ascending=[False, False, True])
    b = target_B_full[target_B_full.user_id == u].copy()
    b = b.sort_values(["n_watch", "last_ts", "tv_show_id"],
                      ascending=[False, False, True])
    a_ids   = {int(i) for i in h.tv_show_id}
    b_order = [int(i) for i in b.tv_show_id]
    b_ids   = set(b_order)
    tgt     = [int(i) for i in target_B[u]]

    print("\n" + "=" * 112)
    print(f"USER {u}")
    print("=" * 112)

    if not a_ids:
        print("A history (weeks 11-20): empty -> every method returns the shared fallback")
    else:
        n_a = len(a_ids)
        print(f"A history (weeks 11-20): {n_a} programme{'s' if n_a != 1 else ''}, "
              f"{A_VIEWS.get(u, 0)} airings; top 10 by watch count")
        rows = []
        for r, (_, row) in enumerate(h.head(10).iterrows(), 1):
            i = int(row.tv_show_id)
            t, c = item_label(i)
            rows.append(dict(rank=r, tv_show_id=i, title=t[:44], category=c,
                             airings=int(row.n_watch),
                             last_view=pd.to_datetime(int(row.last_ts), unit="s").date()))
        print(pd.DataFrame(rows).to_string(index=False))

    print(f"\nActually watched in B (weeks 21-30): {len(b)} programmes; top {min(10, len(b))} shown")
    rows = []
    for r, (_, row) in enumerate(b.head(10).iterrows(), 1):
        i = int(row.tv_show_id)
        t, c = item_label(i)
        rows.append(dict(rank=r, tv_show_id=i, title=t[:44], category=c,
                         airings=int(row.n_watch),
                         scored_target=(f"#{tgt.index(i)+1}" if i in tgt else "")))
    print(pd.DataFrame(rows).to_string(index=False))

    print("\nranked lists (rank 1 = top recommendation)")
    for short, long, pred in METHOD_PREDS:
        ids = [int(i) for i in pred[u]]
        assert len(ids) == K, f"{short}: {len(ids)} ids, expected {K}"
        rows = []
        for r, i in enumerate(ids, 1):
            t, c = item_label(i)
            rows.append(dict(rank=r, tv_show_id=i, title=t[:44], category=c,
                             in_A_history=("yes" if i in a_ids else ""),
                             in_B_target=(f"hit #{b_order.index(i)+1}" if i in b_ids else "")))
        print(f"\n  {long}   [{short}]")
        print(pd.DataFrame(rows).to_string(index=False))
        print("  ids: " + " ".join(str(i) for i in ids))

    print("\n  side by side (ids only; rank reads down)")
    side = pd.DataFrame({"rank": range(1, K + 1)})
    for short, _long, pred in METHOD_PREDS:
        side[short] = [int(i) for i in pred[u]]
    print(side.to_string(index=False))

    print("\n  against the actual B target")
    for short, _long, pred in METHOD_PREDS:
        got  = [int(i) for i in pred[u]]
        hits = [i for i in got if i in b_ids]
        rks  = [b_order.index(i) + 1 for i in hits]
        print(f"    {short:10s} {len(hits)}/{K} target items | target ranks {rks or '-'} | "
              f"familiar {sum(i in a_ids for i in got)}/{K} already in A")


for u in EXAMPLE_USERS:
    show_user(u)

example users
  13866173795451203812  warm               | A programmes 262 | A airings 1358 | B target 5
  223991263617468882  warm               | A programmes   1 | A airings    2 | B target 5
  cold validation ids available for an example: [414363484269016605, 665715564913042433, 755371787188216785, 970670590731358344, 979634918703198026, 1002957187089808415, 1524839109561562221, 3468053194041632395] ...
  set EXAMPLE_USERS = [13866173795451203812, 414363484269016605] to study one
  shared fallback (identical for every method): [2400480, 240081, 2400467, 200344, 2400508]
  output contract: every list below has exactly 5 ids; a short list is padded from FALLBACK

USER 13866173795451203812
A history (weeks 11-20): 262 programmes, 1358 airings; top 10 by watch count
 rank  tv_show_id                                        title    category  airings  last_view
    1     2400480                                         ТСН.        Инфо      157 2020-05-03
    2      240081               

## Step 8 — Business and algorithmic analysis

The cell below produces the numbers the discussion needs; the prose is written from those numbers after a run, not from assumptions.

### 8.1 What it measures

* **Novelty per method** — the share of recommended slots that were *not* in the user's A history. High novelty is not automatically good: the user may simply never want the programme.
* **Target reach** — mean number of the five actual B items a method puts in its top-5, the share of users where the list equals the target set, and the mean rank of the first hit (6 when there is no hit). The hits are counted against the five-item target `target_B[u]`, not against every programme watched in B. This is *not* MAP@5; MAP@5 is in `comparison`, and it weights the hit position.
* **Agreement** — for the two example users, how many ids content and PPR share, how many the hybrid shares with each component, and how many each shares with the user's own-history list. If two methods agree on every slot they are not offering different products.
* **Platform limits** — the duration-weighted share of scheduled airtime that carries an id, and the share of validation target items that has at least one A view (the only items PageRank can reach at all).

### 8.2 The questions to answer with it

1. **Familiar or new?** Compare each method's `new_to_user_share` (§8.1) with the target's new share, 46.9 % (100 % minus the 53.1 % repeat share of the actual target). A method that is far more novel than the target is probably recommending programmes the user did not watch; one that is far less novel is just replaying history.
2. **Why the methods differ.** Content can score a programme through shared title words, genre, cast or decade even when nobody watched it; PageRank cannot, because an item with no A views has no co-view edges and zero mass. Conversely, PageRank uses the fact that people watch programmes in company — a relation no metadata field records — and its diffusion step re-weights the whole neighbourhood of the user's history, so a strongly co-viewed programme can outrank a closer metadata match. Both statements are testable in the per-user tables: look for slots where one method has a hit and the other does not. *Scope note:* this notebook implements content similarity and co-view diffusion; it does not implement association rules or matrix factorisation, so the generic "rules vs MF" contrast is outside the evidence here. The comparison that the results actually support is **metadata similarity vs co-view graph**.
3. **Where popularity already does well.** Compare the popularity row with the example users' lists. For a light viewer with a thin history, a global list is hard to beat because there is little personal signal; the `cold_slot_hit` and `fallback_users` columns show how much of the population is in that situation.
4. **Why the shipped strategy is preferable.** Use `comparison` (validation MAP@5, warm and all), the `seconds` columns (cost at this scale), and the example lists (what the user sees). The chosen candidate is the one with the highest hold-out MAP@5 under the rule in §5.1; the component files are kept so the claim, and the fusion itself, can be checked.
5. **Limitations the results support.** Start with the labelled-airtime share printed below — most airtime has no id and can never become an interaction. In this run 39.4 % of B target items have no A view, so PageRank cannot reach them. Then the 83.6 % tie mass at the fifth frequency value (§2.4), the test-only programmes PageRank cannot reach, and the fact that an empty log means "not recorded on this platform", not "never seen".

### 8.3 Cautions to state explicitly in the write-up

* An unfamiliar programme is not automatically a useful discovery. Nothing in this log records enjoyment, so novelty and relevance are different claims.
* Absence from the log does not establish that a user has never seen a programme; the log covers this platform only.
* Words like "never", "always" and "better" must be qualified to the weeks, population and metric examined here.
* The hold-out tie structure means small MAP differences are inside label noise; the example lists illustrate behaviour, they do not prove a winner.

In [62]:
# ==============================================================
# 8.1  Evidence for the analysis: novelty, target reach, agreement, limits
# ==============================================================
A_BY_USER = {int(u): set(map(int, s)) for u, s in hist_A.groupby("user_id").tv_show_id}

rows = []
for name, pred in (("content-based (method 1)", pred_content),
                   ("personalised PageRank (method 2)", pred_ppr),
                   ("hybrid", pred_hybrid),
                   ("popularity (reference)", pred_pop),
                   ("own A-history top-5 (reference)", pred_own)):
    novel = hits = exact = first = 0
    for u in WARM_TARGET_USERS:
        got  = [int(i) for i in pred[u]]
        seen = A_BY_USER.get(u, set())
        tg   = set(target_B[u])
        novel += sum(i not in seen for i in got)
        hits  += len(set(got) & tg)
        exact += (set(got) == tg)
        ranks  = [r for r, i in enumerate(got, 1) if i in tg]
        first += ranks[0] if ranks else K + 1
    n = len(WARM_TARGET_USERS)
    rows.append(dict(method=name,
                     new_to_user_share=round(novel / (n * K), 3),
                     target_hits_per_user=round(hits / n, 2),
                     exact_5_of_5=round(exact / n, 3),
                     mean_first_hit_rank=round(first / n, 2)))

analysis_evidence = pd.DataFrame(rows)
print(f"aggregate over the {len(WARM_TARGET_USERS):,} warm validation users")
print(analysis_evidence.to_string(index=False))
print("\nnew_to_user_share = slots not in A history; target_hits_per_user is out of 5;")
print("mean_first_hit_rank uses 6 when there is no hit. MAP@5 is the metric in `comparison`.")

print("\nlist agreement for the two example users")
for u in EXAMPLE_USERS:
    c  = [int(i) for i in pred_content[u]]
    p  = [int(i) for i in pred_ppr[u]]
    h  = [int(i) for i in pred_hybrid[u]]
    o  = [int(i) for i in pred_own[u]]
    tg = set(target_B[u])
    print(f"  {u}: content∩PPR {len(set(c) & set(p))}/5 | "
          f"hybrid∩PPR {len(set(h) & set(p))}/5 | hybrid∩content {len(set(h) & set(c))}/5 | "
          f"content∩own {len(set(c) & set(o))}/5 | PPR∩own {len(set(p) & set(o))}/5 | "
          f"content hits {len(set(c) & tg)} | PPR hits {len(set(p) & tg)} | "
          f"hybrid hits {len(set(h) & tg)}")

lab = sched[sched.tv_show_id != 0]
print(f"\nlabelled airtime     : {lab.duration.sum() / sched.duration.sum():.1%} of scheduled "
      f"seconds ({len(lab) / len(sched):.1%} of slots) -> most airtime cannot become an interaction")
tgt_items = {int(i) for ids in target_B.values() for i in ids}
reachable = {int(i) for i in hist_A.tv_show_id}
print(f"B target items with >=1 A view (co-view reachable): "
      f"{len(tgt_items & reachable) / len(tgt_items):.1%}")
print(f"B target items with zero A views (PPR cannot reach): "
      f"{len(tgt_items - reachable) / len(tgt_items):.1%}")

print("\nhow to read this:")
print("  * novelty far above the target's 53.1% repeat share = likely misses, not discovery")
print("  * an id in one list and not the other is the concrete difference to discuss in Step 8.2")
print("  * a cold example user returns the same fallback list from every method, by design")

aggregate over the 3,868 warm validation users
                          method  new_to_user_share  target_hits_per_user  exact_5_of_5  mean_first_hit_rank
        content-based (method 1)              0.357                  1.61         0.000                 2.15
personalised PageRank (method 2)              0.026                  1.96         0.011                 1.99
                          hybrid              0.078                  2.02         0.014                 1.96
          popularity (reference)              0.364                  1.46         0.000                 2.52
 own A-history top-5 (reference)              0.025                  1.91         0.011                 2.05

new_to_user_share = slots not in A history; target_hits_per_user is out of 5;
mean_first_hit_rank uses 6 when there is no hit. MAP@5 is the metric in `comparison`.

list agreement for the two example users
  13866173795451203812: content∩PPR 4/5 | hybrid∩PPR 4/5 | hybrid∩content 4/5 | content∩own 